## Condition Configuration

In [ ]:
CONDITION = 'minimal'
N_BOOT = 1000
BOOT_SEED = 42
N_SUBSAMPLE = 23
REUSE_CACHED_EMBEDDINGS = True


# Compare Human vs All-AI Research Proposals (Pooled) — Style-Controlled (Rephrased)

# Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency, fisher_exact
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_distances
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
import pickle
import json
import os
import sys
import warnings
import textwrap
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Find project root ──────────────────────────────────────────────────────
def find_project_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for _ in range(10):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        p = p.parent
    raise FileNotFoundError('Could not find project root with src/ and data/')

try:
    _nb_path = Path(__file__).resolve().parent
except NameError:
    _nb_path = Path.cwd()
PROJECT_ROOT = find_project_root(_nb_path)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

condition = 'rephrased/minimal'
TABLES_DIR    = PROJECT_ROOT / 'results' / 'tables' / condition
FIGURES_DIR   = PROJECT_ROOT / 'results' / 'figures' / condition
TABLES_DIR_AI  = TABLES_DIR / 'all_ai'
FIGURES_DIR_AI = FIGURES_DIR / 'all_ai'
CACHE_DIR     = TABLES_DIR / 'cached'

TABLES_DIR_AI.mkdir(parents=True, exist_ok=True)
FIGURES_DIR_AI.mkdir(parents=True, exist_ok=True)
print(f'TABLES_DIR_AI:  {TABLES_DIR_AI}')
print(f'FIGURES_DIR_AI: {FIGURES_DIR_AI}')

MODEL_NAME_MAP = {
    'GPT-5.2': 'GPT-5.2', 'Gemini': 'Gemini', 'Claude': 'Claude',
    'gpt': 'GPT-5.2', 'gemini': 'Gemini', 'claude': 'Claude',
    'GPT': 'GPT-5.2'
}

# ── Color scheme ───────────────────────────────────────────────────────────
colors = {'Human': '#DC143C', 'All AI': '#4A90E2'}
AI_COLOR    = '#4A90E2'
HUMAN_COLOR = '#DC143C'
HUMAN_FUNDED_COLOR    = HUMAN_COLOR      # legacy alias; funding is shown with a magenta ring
HUMAN_NONFUNDED_COLOR = HUMAN_COLOR      # legacy alias; nonfunded Human keeps base fill

# ── Helper functions ───────────────────────────────────────────────────────
def bootstrap_mean_ci(values, n_boot=2000, random_state=42, alpha=0.05):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0: return np.nan, np.nan, np.nan
    mean = float(np.mean(arr))
    if len(arr) == 1: return mean, mean, mean
    rng = np.random.default_rng(random_state)
    boot_means = np.array([np.mean(rng.choice(arr, size=len(arr), replace=True))
                            for _ in range(n_boot)])
    lo, hi = np.quantile(boot_means, [alpha/2, 1-alpha/2])
    return mean, float(lo), float(hi)

def boot_ci(values, alpha=0.05):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0: return np.nan, np.nan
    return float(np.quantile(arr, alpha/2)), float(np.quantile(arr, 1-alpha/2))

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    dom = sum(1 if xi > yj else (-1 if xi < yj else 0)
              for xi in x for yj in y)
    return dom / (len(x) * len(y))

def bootstrap_cliffs_delta_ci(x, y, n_boot=2000, random_state=42, alpha=0.05):
    rng = np.random.default_rng(random_state)
    x, y = np.asarray(x), np.asarray(y)
    boots = [cliffs_delta(rng.choice(x, len(x), replace=True),
                          rng.choice(y, len(y), replace=True))
             for _ in range(n_boot)]
    return float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1-alpha/2))

def permutation_test(group1, group2, n_permutations=10000, random_state=42):
    rng = np.random.default_rng(random_state)
    g1, g2 = np.asarray(group1), np.asarray(group2)
    obs = np.mean(g1) - np.mean(g2)
    combined = np.concatenate([g1, g2])
    n1 = len(g1)
    perm_diffs = []
    for _ in range(n_permutations):
        p = rng.permutation(combined)
        perm_diffs.append(np.mean(p[:n1]) - np.mean(p[n1:]))
    perm_diffs = np.array(perm_diffs)
    pval = (np.sum(np.abs(perm_diffs) >= abs(obs)) + 1) / (n_permutations + 1)
    return float(pval), float(obs), perm_diffs

def fmt_p(p):
    if np.isnan(p): return 'n/a'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return f'p={p:.3f}'

def group_mst_dispersion(idx, D):
    Dg = D[np.ix_(idx, idx)]
    if len(idx) <= 1: return 0.0
    mst = minimum_spanning_tree(Dg)
    return float(mst.sum() / max(1, len(idx)-1))

def group_chamfer(idx, D):
    Dg = D[np.ix_(idx, idx)].copy()
    np.fill_diagonal(Dg, np.inf)
    return float(np.min(Dg, axis=1).mean())

def group_sparseness(idx, D):
    Dg = D[np.ix_(idx, idx)]
    med_i = int(np.argmin(Dg.sum(axis=0)))
    return float(Dg[:, med_i].mean())

def group_mean_pairwise(idx, D):
    Dg = D[np.ix_(idx, idx)].copy()
    np.fill_diagonal(Dg, np.nan)
    return float(np.nanmean(Dg))

def group_loo_centroid(idx, X):
    Xg = X[idx]
    n = len(idx)
    if n <= 1: return np.zeros(n)
    s = Xg.sum(axis=0)
    result = []
    for i in range(n):
        centroid_loo = (s - Xg[i]) / (n - 1)
        d = cosine_distances(Xg[i:i+1], centroid_loo.reshape(1, -1))[0, 0]
        result.append(d)
    return np.array(result)

def group_global_centroid(idx, X):
    c = X.mean(axis=0, keepdims=True)
    return cosine_distances(X[idx], c).ravel()

def group_grid_entropy(idx, coords, bins=5):
    pts = coords[idx]
    H, _, _ = np.histogram2d(pts[:, 0], pts[:, 1], bins=bins)
    p = H.ravel().astype(float)
    p = p[p > 0]
    p = p / p.sum()
    ent = -np.sum(p * np.log(p))
    return float(ent / np.log(bins * bins)) if np.log(bins * bins) > 0 else 0.0

def per_proposal_pairwise(idx, D):
    Dg = D[np.ix_(idx, idx)].copy()
    np.fill_diagonal(Dg, np.nan)
    return np.nanmean(Dg, axis=1)

def per_proposal_nn(idx, D):
    Dg = D[np.ix_(idx, idx)].copy()
    np.fill_diagonal(Dg, np.inf)
    return np.min(Dg, axis=1)

def per_proposal_medoid_dist(idx, D):
    Dg = D[np.ix_(idx, idx)]
    med_i = int(np.argmin(Dg.sum(axis=0)))
    return Dg[:, med_i]

def per_proposal_loo_centroid(idx, X):
    return group_loo_centroid(idx, X)

def boot_metric_over_samples(metric_fn, boot_samples, *args):
    return np.array([metric_fn(s, *args) for s in boot_samples])

def add_mean_ci_to_boxplot(ax, data_arrays, positions=None, colors_for_points=None,
                            n_boot=2000, random_state=42, marker_size=50,
                            line_color='black', zorder=12):
    if positions is None:
        positions = np.arange(len(data_arrays))
    if colors_for_points is None:
        colors_for_points = ['#808080'] * len(data_arrays)
    for pos, values, pc in zip(positions, data_arrays, colors_for_points):
        mean, lo, hi = bootstrap_mean_ci(values, n_boot=n_boot, random_state=random_state)
        if not np.isfinite(mean):
            continue
        yerr = np.array([[max(0., mean - lo)], [max(0., hi - mean)]])
        rgba = mcolors.to_rgba(pc, 0.95)
        ax.errorbar(pos, mean, yerr=yerr, fmt='D',
                    markersize=np.sqrt(marker_size),
                    markerfacecolor=rgba, markeredgecolor=line_color,
                    markeredgewidth=1.2, ecolor=line_color,
                    elinewidth=1.4, capsize=4, capthick=1.2, zorder=zorder)

FUNDED_HUMAN_RING_COLOR = '#FF00FF'


def _coerce_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {'1', 'true', 'yes', 'y'}
    return bool(value)


def _augment_point_metadata(meta):
    if meta is None:
        return None
    df = meta.copy()
    if 'proposal_uid' in df.columns and 'proposal_meta' in globals():
        add_cols = [c for c in ['proposal_uid', 'funding', 'is_top5_ranked', 'group_binary'] if c in proposal_meta.columns]
        base = proposal_meta[add_cols].drop_duplicates('proposal_uid')
        for col in add_cols:
            if col == 'proposal_uid' or col in df.columns:
                continue
            df = df.merge(base[['proposal_uid', col]], on='proposal_uid', how='left')
    return df.reset_index(drop=True)


def _is_funded_human(row, fallback_group):
    group_binary = str(row.get('group_binary', fallback_group))
    is_human = (str(fallback_group).startswith('Human')) or (group_binary == 'Human')
    if not is_human:
        return False
    funding = pd.to_numeric(row.get('funding', np.nan), errors='coerce')
    if np.isfinite(funding):
        return funding == 1
    funding_txt = str(row.get('funding', '')).strip().lower()
    return funding_txt in {'funded', 'yes', 'true'}


def _metadata_legend_handles(include_funding=True, include_top5=True):
    handles = []
    if include_funding:
        handles.append(mlines.Line2D([], [], marker='o', linestyle='None', markersize=7,
                                     markerfacecolor='white', markeredgecolor=FUNDED_HUMAN_RING_COLOR,
                                     markeredgewidth=1.7, label='Funded human'))
    if include_top5:
        handles.append(mlines.Line2D([], [], marker='o', linestyle='None', markersize=8,
                                     markerfacecolor='white', markeredgecolor='black',
                                     markeredgewidth=1.5, label='Top-ranked proposal'))
    return handles


def draw_boxplot_with_jitter(ax, data_list, labels, group_colors, positions=None,
                              jitter=0.15, point_size=20, point_alpha=0.5,
                              box_alpha=0.7, point_meta_list=None,
                              show_metadata_legend=False, metadata_legend_loc='best'):
    if positions is None:
        positions = list(range(len(data_list)))
    rng_j = np.random.default_rng(0)
    bp = ax.boxplot(data_list, positions=positions, widths=0.4,
                    patch_artist=True, showfliers=True,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(color='black', linewidth=1.2),
                    capprops=dict(color='black', linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, markerfacecolor='white',
                                    markeredgecolor='black', alpha=0.55, linestyle='none'))
    for patch, col in zip(bp['boxes'], group_colors):
        patch.set_facecolor(mcolors.to_rgba(col, box_alpha))
        patch.set_edgecolor('black')
        patch.set_linewidth(1.2)
    if point_meta_list is None:
        point_meta_list = [None] * len(data_list)
    has_metadata = False
    for data, pos, col, label, meta in zip(data_list, positions, group_colors, labels, point_meta_list):
        data = np.asarray(data, dtype=float)
        xs = rng_j.uniform(pos - jitter, pos + jitter, size=len(data))
        meta_df = _augment_point_metadata(meta)
        if meta_df is not None and len(meta_df) == len(data):
            has_metadata = True
        else:
            meta_df = None
        ax.scatter(xs, data, s=point_size, color=col, alpha=point_alpha, edgecolors='none', zorder=5)
        if meta_df is not None:
            for x_i, y_i, (_, row) in zip(xs, data, meta_df.iterrows()):
                if not np.isfinite(y_i):
                    continue
                if _is_funded_human(row, label):
                    ax.scatter(x_i, y_i, s=point_size + 34, facecolors='none',
                               edgecolors=FUNDED_HUMAN_RING_COLOR, linewidths=1.7, zorder=6)
                if _coerce_bool(row.get('is_top5_ranked', False)):
                    ax.scatter(x_i, y_i, s=point_size + 68, facecolors='none',
                               edgecolors='black', linewidths=1.5, zorder=6.3)
    add_mean_ci_to_boxplot(ax, data_list, positions=positions,
                            colors_for_points=group_colors)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.yaxis.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    if show_metadata_legend and has_metadata:
        handles = _metadata_legend_handles(include_funding=True, include_top5=True)
        if handles:
            ax.legend(handles=handles, fontsize=8, loc=metadata_legend_loc, framealpha=0.88)
    return bp


def _proposal_visual_arrays(meta=None):
    """Return top-rank, funded-human, group, and marker-size arrays aligned to proposal rows."""
    if meta is None:
        meta = proposal_meta
    n = len(meta)
    if 'is_top5_ranked' in meta.columns:
        top5 = meta['is_top5_ranked'].fillna(False).map(_coerce_bool).to_numpy(dtype=bool)
    else:
        top5 = np.zeros(n, dtype=bool)

    if 'funding' in meta.columns:
        funding_num = pd.to_numeric(meta['funding'], errors='coerce')
        funding_txt = meta['funding'].astype(str).str.strip().str.lower()
        funded = (funding_num == 1).fillna(False).to_numpy(dtype=bool) | funding_txt.isin(['funded', 'yes', 'true', '1']).to_numpy(dtype=bool)
    else:
        funded = np.zeros(n, dtype=bool)

    if 'group_binary' in meta.columns:
        group_vals = meta['group_binary'].astype(str).to_numpy()
    elif 'group_model' in meta.columns:
        group_vals = meta['group_model'].astype(str).to_numpy()
    else:
        group_vals = np.array([''] * n, dtype=object)

    if 'viz_marker_size' in meta.columns:
        sizes = pd.to_numeric(meta['viz_marker_size'], errors='coerce').fillna(75).to_numpy(dtype=float)
    else:
        sizes = np.full(n, 75.0)
    return top5, funded, group_vals, sizes


def scatter_proposal_points(ax, coords, mask, color, label, marker='o', alpha=0.86,
                            size=58, zorder=5, meta=None, show_label=True):
    """Draw proposal points with shared metadata rings: magenta funded-Human, black top-ranked."""
    if meta is None:
        meta = proposal_meta
    coords = np.asarray(coords)
    mask = np.asarray(mask, dtype=bool)
    if coords.shape[0] != len(meta) or len(mask) != len(meta):
        raise ValueError('coords, mask, and proposal metadata must be aligned row-wise')
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return []

    top5, funded, group_vals, default_sizes = _proposal_visual_arrays(meta)
    if np.isscalar(size):
        base_sizes = np.full(len(meta), float(size))
    else:
        base_sizes = np.asarray(size, dtype=float)
        if len(base_sizes) != len(meta):
            base_sizes = default_sizes

    handle = ax.scatter(coords[idx, 0], coords[idx, 1], c=color, s=base_sizes[idx], marker=marker,
                        alpha=alpha, edgecolors='none', linewidths=0,
                        label=(label if show_label else None), zorder=zorder)

    funded_idx = idx[(group_vals[idx] == 'Human') & funded[idx]]
    if len(funded_idx):
        ax.scatter(coords[funded_idx, 0], coords[funded_idx, 1], s=base_sizes[funded_idx] + 46,
                   marker=marker, facecolors='none', edgecolors=FUNDED_HUMAN_RING_COLOR,
                   linewidths=1.8, alpha=0.95, label=None, zorder=zorder + 0.25)

    top_idx = idx[top5[idx]]
    if len(top_idx):
        ax.scatter(coords[top_idx, 0], coords[top_idx, 1], s=base_sizes[top_idx] + 92,
                   marker=marker, facecolors='none', edgecolors='black',
                   linewidths=1.8, alpha=0.96, label=None, zorder=zorder + 0.5)
    return [handle]


def proposal_group_legend_handles(include_human=True, include_ai=True, human_label='Human', ai_label='All AI'):
    handles = []
    if include_human:
        handles.append(mlines.Line2D([], [], marker='o', linestyle='None', markersize=7,
                                     markerfacecolor=HUMAN_COLOR, markeredgecolor='none',
                                     label=human_label))
    if include_ai:
        handles.append(mlines.Line2D([], [], marker='D', linestyle='None', markersize=7,
                                     markerfacecolor=AI_COLOR, markeredgecolor='none',
                                     label=ai_label))
    return handles


def add_proposal_metadata_legend(ax, group_handles=None, loc='best', title='Proposals', fontsize=8):
    handles = list(group_handles or []) + _metadata_legend_handles(include_funding=True, include_top5=True)
    if handles:
        ax.legend(handles=handles, loc=loc, fontsize=fontsize, title=title,
                  title_fontsize=fontsize + 1, framealpha=0.9)


def draw_effect_size_panel(ax, delta, d_lo, d_hi, label='All AI', p_val=None):
    ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
    ax.errorbar(delta, 0, xerr=[[delta - d_lo], [d_hi - delta]],
                fmt='o', color=AI_COLOR, markersize=8,
                ecolor='black', elinewidth=1.5, capsize=5, capthick=1.5)
    ax.set_yticks([0])
    ax.set_yticklabels([label])
    ax.set_xlabel("Cliff's δ (bootstrap 95% CI)\npositive = AI > Human")
    ax.yaxis.grid(True, alpha=0.25)
    if p_val is not None:
        ax.set_title(fmt_p(p_val), fontsize=10)

print('Imports and helpers loaded.')


## Load Shared Precomputed Caches

In [ ]:
# ── Load .npy caches ────────────────────────────────────────────────────────
def _load_npy(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Required cache missing: {p}')
    return np.load(str(p))

D_pp    = _load_npy(CACHE_DIR / 'proposal_distance_matrix.npy')   # (92,92)
X_pca2d = _load_npy(CACHE_DIR / 'proposal_pca2d.npy')             # (92,2)
X_umap2d= _load_npy(CACHE_DIR / 'proposal_umap2d.npy')            # (92,2)
print(f'D_pp: {D_pp.shape}, X_pca2d: {X_pca2d.shape}, X_umap2d: {X_umap2d.shape}')

# Literature SELF-kNN distances (39538×50) — used for novelty_z local density.
# NOTE: this is lit-to-lit kNN, NOT proposal-to-literature distances.
_lit_knn_path = CACHE_DIR / 'lit_knn_distances_50.npy'
lit_knn_distances_50 = np.load(str(_lit_knn_path)) if _lit_knn_path.exists() else None
if lit_knn_distances_50 is not None:
    print(f'lit_knn_distances_50 (lit-lit self-kNN): {lit_knn_distances_50.shape}')
    lit_mean_knn_10 = lit_knn_distances_50[:, :10].mean(axis=1)
else:
    print('WARNING: lit_knn_distances_50.npy not found — novelty_z normalization unavailable')
    lit_mean_knn_10 = None

# ── Proposal embeddings (full-proposal, in proposal_meta row order) ──────────
# pkl stores {'human_embeddings': ..., 'ai_embeddings': ..., ...}
# proposal_meta rows are ordered: humans first (matching vstack order in original notebook)
_emb_path = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_human_ai_rephrased.pkl'
with open(_emb_path, 'rb') as f:
    _emb_data = pickle.load(f)
if isinstance(_emb_data, dict) and 'human_embeddings' in _emb_data:
    _h_emb = np.asarray(_emb_data['human_embeddings'], dtype=np.float32)
    _a_emb = np.asarray(_emb_data['ai_embeddings'],    dtype=np.float32)
    X_prop = np.vstack([_h_emb, _a_emb])
else:
    X_prop = np.asarray(_emb_data, dtype=np.float32)
_norms = np.linalg.norm(X_prop, axis=1, keepdims=True); _norms[_norms == 0] = 1
X_prop = (X_prop / _norms).astype(np.float32)
print(f'X_prop: {X_prop.shape}')

# ── Literature embeddings + proposal-to-literature distances ─────────────────
# D_pl_sorted_idx  (92 × n_lit) and D_pl_sorted_dist (92 × 50) are computed here.
# They are NOT stored in the shared cache; they must be computed from embeddings.
_lit_emb_path = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'relevant_literature_embeddings.pkl'
_abs_emb_path = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_rephrased_abstract.pkl'

D_pl_sorted_idx  = None
D_pl_sorted_dist = None
X_lit = None

if _lit_emb_path.exists():
    print('Loading literature embeddings...')
    with open(_lit_emb_path, 'rb') as f:
        _lit_raw = pickle.load(f)
    if isinstance(_lit_raw, dict) and 'embeddings' in _lit_raw:
        X_lit = np.asarray(_lit_raw['embeddings'], dtype=np.float32)
    elif isinstance(_lit_raw, dict):
        X_lit = np.vstack(list(_lit_raw.values())).astype(np.float32)
    else:
        X_lit = np.asarray(_lit_raw, dtype=np.float32)
    _ln = np.linalg.norm(X_lit, axis=1, keepdims=True); _ln[_ln == 0] = 1
    X_lit = (X_lit / _ln).astype(np.float32)
    print(f'X_lit: {X_lit.shape}')

    # Use abstract-only proposal embeddings for novelty (matches original notebook)
    X_prop_nov = X_prop
    if _abs_emb_path.exists():
        with open(_abs_emb_path, 'rb') as f:
            _abs_raw = pickle.load(f)
        if isinstance(_abs_raw, dict) and 'human_embeddings' in _abs_raw:
            _ha = np.asarray(_abs_raw['human_embeddings'], dtype=np.float32)
            _aa = np.asarray(_abs_raw['ai_embeddings'],    dtype=np.float32)
            _X_abs = np.vstack([_ha, _aa])
        elif isinstance(_abs_raw, dict) and 'embeddings' in _abs_raw:
            _X_abs = np.asarray(_abs_raw['embeddings'], dtype=np.float32)
        else:
            _X_abs = np.asarray(_abs_raw, dtype=np.float32)
        if _X_abs.shape[0] == X_prop.shape[0]:
            _an = np.linalg.norm(_X_abs, axis=1, keepdims=True); _an[_an == 0] = 1
            X_prop_nov = (_X_abs / _an).astype(np.float32)
            print(f'X_prop_nov (abstract-only): {X_prop_nov.shape}')
        else:
            print(f'WARNING: abstract emb shape mismatch, falling back to full-proposal emb')

    print('Computing proposal-to-literature distances (92 × 39538)...')
    S_pl = X_prop_nov @ X_lit.T                                        # (92, n_lit)
    D_pl = (1.0 - S_pl).astype(np.float32)
    D_pl_sorted_idx  = np.argsort(D_pl, axis=1)                        # (92, n_lit)
    D_pl_sorted_dist = np.take_along_axis(D_pl, D_pl_sorted_idx, axis=1)[:, :50]  # (92, 50)
    print(f'D_pl_sorted_idx: {D_pl_sorted_idx.shape}, D_pl_sorted_dist: {D_pl_sorted_dist.shape}')
    del S_pl, D_pl
else:
    print('WARNING: literature embeddings not found — novelty/MeSH/BERTopic recomputation skipped')

# ── Proposal metadata ─────────────────────────────────────────────────────────
# Prefer cached version which has full columns (group_binary, proposal_uid, ranking, funding, etc.)
_meta_path = CACHE_DIR / 'proposal_meta.csv'
if not _meta_path.exists():
    _meta_path = TABLES_DIR / 'proposal_metadata.csv'
proposal_meta = pd.read_csv(_meta_path)
# Derive missing columns if loaded from the minimal CSV (only has 'group')
if 'group' in proposal_meta.columns and 'group_binary' not in proposal_meta.columns:
    proposal_meta['group_binary'] = proposal_meta['group'].apply(
        lambda x: 'Human' if x == 'Human' else 'AI')
if 'group' in proposal_meta.columns and 'group_model' not in proposal_meta.columns:
    proposal_meta['group_model'] = proposal_meta['group'].map(
        lambda x: MODEL_NAME_MAP.get(x, x))
if 'proposal_uid' not in proposal_meta.columns:
    proposal_meta['proposal_uid'] = [f'P_{i:03d}' for i in range(len(proposal_meta))]
if 'group_model' in proposal_meta.columns:
    proposal_meta['group_model'] = proposal_meta['group_model'].map(lambda x: MODEL_NAME_MAP.get(x, x))
if 'is_top5_ranked' not in proposal_meta.columns:
    if 'ranking' in proposal_meta.columns:
        proposal_meta['is_top5_ranked'] = proposal_meta['ranking'].apply(
            lambda x: pd.notna(x) and float(x) <= 5 if pd.notna(x) else False)
    else:
        proposal_meta['is_top5_ranked'] = False
# Keep a legacy color column neutral; funding status is visualized with magenta rings.
proposal_meta['human_color_shade'] = HUMAN_COLOR
print(f'proposal_meta: {proposal_meta.shape}')
print(proposal_meta['group_binary'].value_counts() if 'group_binary' in proposal_meta.columns else 'no group_binary col')

# ── Index arrays ──────────────────────────────────────────────────────────────
human_idx  = proposal_meta.index[proposal_meta['group_binary'] == 'Human'].to_numpy()
ai_idx     = proposal_meta.index[proposal_meta['group_binary'] == 'AI'].to_numpy()
claude_idx = proposal_meta.index[proposal_meta['group_model'] == 'Claude'].to_numpy()
gemini_idx = proposal_meta.index[proposal_meta['group_model'] == 'Gemini'].to_numpy()
gpt_idx    = proposal_meta.index[proposal_meta['group_model'] == 'GPT-5.2'].to_numpy()
print(f'human_idx: {len(human_idx)}, ai_idx: {len(ai_idx)}')
print(f'claude: {len(claude_idx)}, gemini: {len(gemini_idx)}, gpt: {len(gpt_idx)}')

D_pp_infdiag = D_pp.copy()
np.fill_diagonal(D_pp_infdiag, np.inf)

# ── Literature ancillary data ─────────────────────────────────────────────────
_lb_path = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_assignments.csv'
lit_bertopic_df = pd.read_csv(_lb_path) if _lb_path.exists() else None
if lit_bertopic_df is None:
    print('WARNING: lit_bertopic_assignments.csv not found')

_ureducer_path = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'lit_umap_reducer.pkl'
if _ureducer_path.exists():
    with open(_ureducer_path, 'rb') as f:
        lit_umap_reducer = pickle.load(f)
else:
    lit_umap_reducer = None
    print('WARNING: lit_umap_reducer.pkl not found')

_lumap_path = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'lit_umap2d.npy'
lit_umap2d = np.load(str(_lumap_path)) if _lumap_path.exists() else None
if lit_umap2d is None:
    print('WARNING: lit_umap2d.npy not found')

_articles_path = PROJECT_ROOT / 'data' / 'prepared' / condition / 'literature_corpus_prepared.json'
with open(_articles_path) as f:
    _art_raw = json.load(f)
articles = _art_raw.get('articles', _art_raw) if isinstance(_art_raw, dict) else _art_raw
print(f'articles: {len(articles)} items')

_pp_path = PROJECT_ROOT / 'data' / 'prepared' / condition / 'all_proposals.json'
with open(_pp_path) as f:
    prepared_proposals = json.load(f)
print(f'prepared_proposals: {len(prepared_proposals)} items')

# ── Optional precomputed CSVs ──────────────────────────────────────────────
def _try_load_csv(path, varname):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f'WARNING: {varname} not found at {p}')
    return None

novelty_element_df   = _try_load_csv(TABLES_DIR / 'novelty_element_percentiles.csv', 'novelty_element_df')
novelty_knn_df       = _try_load_csv(TABLES_DIR / 'novelty_mean_knn_scores.csv', 'novelty_knn_df')
novelty_norm_df      = _try_load_csv(TABLES_DIR / 'novelty_local_density_normalized.csv', 'novelty_norm_df')
mesh_proposal_df     = _try_load_csv(TABLES_DIR / 'mesh_coverage_per_proposal.csv', 'mesh_proposal_df')
bertopic_proposal_df = _try_load_csv(TABLES_DIR / 'bertopic_region_coverage_per_proposal.csv', 'bertopic_proposal_df')
style_df             = _try_load_csv(TABLES_DIR / 'style_features.csv', 'style_df')
ward_df              = _try_load_csv(TABLES_DIR / 'ward_cluster_bertopic_style_labels.csv', 'ward_df')
lda_labels_df        = _try_load_csv(TABLES_DIR / 'lda_topic_contrastive_labels.csv', 'lda_labels_df')
topic_cluster_df     = _try_load_csv(TABLES_DIR / 'topic_cluster_assignment_labels.csv', 'topic_cluster_df')
gmm_comp_df          = _try_load_csv(TABLES_DIR / 'cluster_gmm_composition_per_model.csv', 'gmm_comp_df')
master_df            = _try_load_csv(TABLES_DIR / 'proposal_metrics_master.csv', 'master_df')

print('\nAll caches loaded.')

# ── Define top-ranked proposal rings for pooled Human vs All-AI views ─────────
# ranking_AI_reviews_global is AI-only: 1-69 across the pooled AI proposals,
# not a mixed Human+AI rank. Use the five smallest rank rows, not rank <= 5.
# Review ranks are average ranks under ties, so the sequence can jump
# (e.g., AI global ranks 1, 2, 3, then 10).
_existing_top5_ranked = proposal_meta['is_top5_ranked'].copy()
proposal_meta['is_top5_ranked'] = False

_human_pm = proposal_meta['group_binary'].eq('Human')
if 'ranking' in proposal_meta.columns:
    _human_ranks = pd.to_numeric(proposal_meta.loc[_human_pm, 'ranking'], errors='coerce')
    _human_top5_idx = _human_ranks.dropna().nsmallest(5).index
    proposal_meta.loc[_human_top5_idx, 'is_top5_ranked'] = True
else:
    proposal_meta.loc[_human_pm, 'is_top5_ranked'] = _existing_top5_ranked.loc[_human_pm]

_global_rank_lut = {
    r.get('title'): r.get('ranking_AI_reviews_global')
    for r in prepared_proposals
    if r.get('is_ai')
}
if _global_rank_lut:
    proposal_meta['ranking_AI_reviews_global'] = pd.to_numeric(
        proposal_meta['title'].map(_global_rank_lut), errors='coerce'
    )
    _ai_pm = ~_human_pm
    _ai_global_ranks = proposal_meta.loc[_ai_pm, 'ranking_AI_reviews_global'].dropna()
    _ai_top5_idx = _ai_global_ranks.nsmallest(5).index
    proposal_meta.loc[_ai_top5_idx, 'is_top5_ranked'] = True
    print(f'  is_top5_ranked reset: human={int(proposal_meta.loc[_human_pm, "is_top5_ranked"].sum())}, '
          f'all_ai={int(proposal_meta.loc[_ai_pm, "is_top5_ranked"].sum())} '
          f'(five smallest pooled AI-only global ranks)')
else:
    proposal_meta.loc[~_human_pm, 'is_top5_ranked'] = _existing_top5_ranked.loc[~_human_pm]
    print('  WARNING: ranking_AI_reviews_global not found in all_proposals.json '
          '— rerun prepare_data_for_analysis.ipynb to regenerate')


## Generate Bootstrap Subsamples (One-Time — Reused by All N-Sensitive Analyses)

In [ ]:
rng_boot = np.random.default_rng(BOOT_SEED)
boot_ai_idx_samples = [
    rng_boot.choice(ai_idx, size=N_SUBSAMPLE, replace=False)
    for _ in range(N_BOOT)
]
np.save(str(TABLES_DIR_AI / 'bootstrap_ai_idx_samples.npy'),
        np.array(boot_ai_idx_samples))

# Diagnostic: model composition variance across samples
model_counts = {'Claude': [], 'Gemini': [], 'GPT-5.2': []}
for s in boot_ai_idx_samples:
    for m, idx_m in [('Claude', claude_idx), ('Gemini', gemini_idx), ('GPT-5.2', gpt_idx)]:
        model_counts[m].append(np.isin(s, idx_m).sum())
print(f'Bootstrap subsamples: {N_BOOT} x n={N_SUBSAMPLE}')
for m, counts in model_counts.items():
    print(f'  {m}: mean={np.mean(counts):.2f} +/- {np.std(counts):.2f} proposals per subsample')


# PREFLIGHT: AI Model Heterogeneity Tests

Tests whether the 3 AI models are homogeneous on primary outcomes.
Results go to a supplementary table only — they do not change primary Human vs All-AI comparisons.

In [ ]:
hetero_rows = []

def _kw_test(values_by_model, metric_name):
    """Run Kruskal-Wallis across 3 model groups."""
    groups = [v for v in values_by_model if len(v) >= 2]
    if len(groups) < 2:
        return {'metric': metric_name, 'H': np.nan, 'p_kw': np.nan, 'n_models': len(groups)}
    H, p = kruskal(*groups)
    return {'metric': metric_name, 'H': float(H), 'p_kw': float(p), 'n_models': len(groups)}

# Per-proposal diversity metrics from D_pp
pp_all_pairwise = np.array([
    np.nanmean(np.where(np.arange(len(D_pp)) != i,
                        D_pp[i], np.nan))
    for i in range(len(D_pp))
])

hetero_rows.append(_kw_test(
    [pp_all_pairwise[claude_idx], pp_all_pairwise[gemini_idx], pp_all_pairwise[gpt_idx]],
    'per_proposal_mean_pairwise_dist'
))

# Per-proposal global NN distance
_D_inf = D_pp.copy(); np.fill_diagonal(_D_inf, np.inf)
pp_nn_global = np.min(_D_inf, axis=1)
hetero_rows.append(_kw_test(
    [pp_nn_global[claude_idx], pp_nn_global[gemini_idx], pp_nn_global[gpt_idx]],
    'per_proposal_nn_dist_global'
))

# Per-proposal global centroid
pp_global_centroid = group_global_centroid(np.arange(len(D_pp)), X_prop)
hetero_rows.append(_kw_test(
    [pp_global_centroid[claude_idx], pp_global_centroid[gemini_idx], pp_global_centroid[gpt_idx]],
    'per_proposal_global_centroid_dist'
))

# LOO centroid (per-model subgroups)
for m, idx_m in [('Claude', claude_idx), ('Gemini', gemini_idx), ('GPT-5.2', gpt_idx)]:
    loo_m = group_loo_centroid(idx_m, X_prop)
    # store for KW across models not meaningful (loo is within-model centroid)

# Novelty metrics
if novelty_element_df is not None:
    for col in ['element_novel_0', 'element_novel_1', 'element_novel_5', 'element_novel_10']:
        if col in novelty_element_df.columns:
            c_vals = novelty_element_df[novelty_element_df['group_model'] == 'Claude'][col].values
            g_vals = novelty_element_df[novelty_element_df['group_model'] == 'Gemini'][col].values
            p_vals = novelty_element_df[novelty_element_df['group_model'] == 'GPT-5.2'][col].values
            hetero_rows.append(_kw_test([c_vals, g_vals, p_vals], f'novelty_{col}'))

if novelty_knn_df is not None:
    for col in ['mean_knn_5', 'mean_knn_10', 'mean_knn_20', 'mean_knn_50']:
        if col in novelty_knn_df.columns:
            c_v = novelty_knn_df[novelty_knn_df['group_model'] == 'Claude'][col].values
            g_v = novelty_knn_df[novelty_knn_df['group_model'] == 'Gemini'][col].values
            p_v = novelty_knn_df[novelty_knn_df['group_model'] == 'GPT-5.2'][col].values
            hetero_rows.append(_kw_test([c_v, g_v, p_v], f'novelty_{col}'))

if mesh_proposal_df is not None and 'unique_mesh_count' in mesh_proposal_df.columns:
    c_m = mesh_proposal_df[mesh_proposal_df['group_model'] == 'Claude']['unique_mesh_count'].values
    g_m = mesh_proposal_df[mesh_proposal_df['group_model'] == 'Gemini']['unique_mesh_count'].values
    p_m = mesh_proposal_df[mesh_proposal_df['group_model'] == 'GPT-5.2']['unique_mesh_count'].values
    hetero_rows.append(_kw_test([c_m, g_m, p_m], 'unique_mesh_count'))

if bertopic_proposal_df is not None:
    for col in ['region_entropy', 'max_region_weight']:
        if col in bertopic_proposal_df.columns:
            c_b = bertopic_proposal_df[bertopic_proposal_df['group_model'] == 'Claude'][col].values
            g_b = bertopic_proposal_df[bertopic_proposal_df['group_model'] == 'Gemini'][col].values
            p_b = bertopic_proposal_df[bertopic_proposal_df['group_model'] == 'GPT-5.2'][col].values
            hetero_rows.append(_kw_test([c_b, g_b, p_b], f'bertopic_{col}'))

# Apply Holm correction
hetero_df = pd.DataFrame(hetero_rows)
valid_mask = hetero_df['p_kw'].notna()
if valid_mask.sum() > 0:
    rej, p_holm, _, _ = multipletests(hetero_df.loc[valid_mask, 'p_kw'].values,
                                        alpha=0.05, method='holm')
    hetero_df.loc[valid_mask, 'p_holm'] = p_holm
    hetero_df.loc[valid_mask, 'rejected_holm'] = rej
else:
    hetero_df['p_holm'] = np.nan
    hetero_df['rejected_holm'] = False

hetero_df.to_csv(TABLES_DIR_AI / 'supplementary_ai_heterogeneity_kruskal_wallis.csv', index=False)
n_sig = hetero_df['rejected_holm'].sum() if 'rejected_holm' in hetero_df.columns else 0
print(f'{int(n_sig)} of {len(hetero_df)} outcomes showed significant between-model '
      f'heterogeneity after Holm correction.')
print(hetero_df[['metric', 'H', 'p_kw', 'p_holm', 'rejected_holm']].to_string())


# PART I: THEMATIC AND CLUSTER ANALYSIS

LDA, Ward, and GMM results are loaded from precomputed CSVs. No refitting.

## Analysis 1.1: LDA Topic Distribution (Human vs All-AI)

In [ ]:
if topic_cluster_df is None:
    print('WARNING: topic_cluster_assignment_labels.csv not loaded — skipping Analysis 1.1')
else:
    # Merge topic assignments with group info
    tc_merged = topic_cluster_df.merge(
        proposal_meta[['title', 'group_binary']], on='title', how='left')
    tc_merged['group_binary'] = tc_merged['group_binary'].fillna('AI')

    lda_col = 'lda_topic' if 'lda_topic' in tc_merged.columns else None
    if lda_col is None:
        print('WARNING: lda_topic column not found in topic_cluster_df')
    else:
        topics = sorted(tc_merged[lda_col].unique())
        human_tc = tc_merged[tc_merged['group_binary'] == 'Human']
        ai_tc    = tc_merged[tc_merged['group_binary'] == 'AI']

        # Build contingency table
        rows_ct = []
        for t in topics:
            rows_ct.append([
                (human_tc[lda_col] == t).sum(),
                (ai_tc[lda_col] == t).sum()
            ])
        contingency = np.array(rows_ct)
        chi2, p_chi2, dof, _ = chi2_contingency(contingency)
        print(f'LDA topic distribution — chi2={chi2:.3f}, df={dof}, p={p_chi2:.4f}')

        # Permutation test
        all_topics_list = list(tc_merged[lda_col])
        all_groups_list = list(tc_merged['group_binary'])
        n_human_tc = len(human_tc)
        def _chi2_stat(labs, grps):
            u_topics = sorted(set(labs))
            h_mask = np.array(grps) == 'Human'
            a_mask = ~h_mask
            ct = np.array([[(np.array(labs)[h_mask] == t).sum(),
                            (np.array(labs)[a_mask] == t).sum()]
                           for t in u_topics])
            return chi2_contingency(ct)[0] if ct.min() >= 0 else np.nan

        rng_perm_lda = np.random.default_rng(42)
        obs_chi2 = chi2
        perm_chi2s = []
        for _ in range(5000):
            perm_grps = rng_perm_lda.permutation(all_groups_list)
            try:
                perm_chi2s.append(_chi2_stat(all_topics_list, perm_grps.tolist()))
            except Exception:
                perm_chi2s.append(np.nan)
        perm_chi2s = np.array([v for v in perm_chi2s if np.isfinite(v)])
        p_perm_lda = (np.sum(perm_chi2s >= obs_chi2) + 1) / (len(perm_chi2s) + 1)
        print(f'Permutation p={p_perm_lda:.4f} ({fmt_p(p_perm_lda)})')

        # Per-topic proportions
        prop_rows = []
        for t in topics:
            prop_rows.append({
                'topic': t,
                'human_count': (human_tc[lda_col] == t).sum(),
                'ai_count':    (ai_tc[lda_col] == t).sum(),
                'human_frac':  (human_tc[lda_col] == t).mean(),
                'ai_frac':     (ai_tc[lda_col] == t).mean(),
            })
        prop_df = pd.DataFrame(prop_rows)
        prop_df['chi2'] = chi2
        prop_df['p_chi2'] = p_chi2
        prop_df['p_perm'] = p_perm_lda
        prop_df.to_csv(TABLES_DIR_AI / 'lda_topic_distribution_human_vs_allai.csv', index=False)
        print(prop_df.to_string())


## Analysis 1.2: Topic Participation (Human vs All-AI)

In [ ]:
if topic_cluster_df is None or master_df is None:
    print('WARNING: topic_cluster_df or master_df not loaded — skipping Analysis 1.2')
else:
    tc_m = topic_cluster_df.merge(
        proposal_meta[['title', 'group_binary']], on='title', how='left')
    tc_m['group_binary'] = tc_m['group_binary'].fillna('AI')
    lda_col = 'lda_topic' if 'lda_topic' in tc_m.columns else None
    if lda_col:
        topics_uniq = sorted(tc_m[lda_col].unique())
        part_rows = []
        for t in topics_uniq:
            human_in = (tc_m[tc_m['group_binary'] == 'Human'][lda_col] == t).astype(int)
            ai_in    = (tc_m[tc_m['group_binary'] == 'AI'][lda_col] == t).astype(int)
            ct = np.array([[human_in.sum(), (1 - human_in).sum()],
                            [ai_in.sum(),   (1 - ai_in).sum()]])
            _, p_f = fisher_exact(ct)
            part_rows.append({'topic': t, 'human_in': int(human_in.sum()),
                              'ai_in': int(ai_in.sum()), 'fisher_p': float(p_f)})
        part_df = pd.DataFrame(part_rows)
        if len(part_df) > 0:
            _, part_df['p_holm'], _, _ = multipletests(part_df['fisher_p'], method='holm')
        part_df.to_csv(TABLES_DIR_AI / 'topic_participation_human_vs_allai.csv', index=False)
        print(part_df.to_string())


## Analysis 1.3: Ward Cluster Membership (Human vs All-AI)

In [ ]:
if topic_cluster_df is None:
    print('WARNING: topic_cluster_df not loaded — skipping Analysis 1.3')
    ward_labels_full = None
else:
    tc_ward = topic_cluster_df.merge(
        proposal_meta[['title', 'group_binary', 'group_model']], on='title', how='left')
    tc_ward['group_binary'] = tc_ward['group_binary'].fillna('AI')

    ward_col = 'ward_cluster_name' if 'ward_cluster_name' in tc_ward.columns else 'ward_cluster_display'
    if ward_col not in tc_ward.columns:
        ward_col = None
        print('WARNING: ward cluster column not found')

    if ward_col:
        human_w = tc_ward[tc_ward['group_binary'] == 'Human']
        ai_w    = tc_ward[tc_ward['group_binary'] == 'AI']
        clusters = sorted(tc_ward[ward_col].dropna().unique())

        # 2x2 contingency (Human / AI) x (Cluster A / Cluster B)
        if len(clusters) == 2:
            ct_ward = np.array([
                [(human_w[ward_col] == clusters[0]).sum(), (human_w[ward_col] == clusters[1]).sum()],
                [(ai_w[ward_col] == clusters[0]).sum(),    (ai_w[ward_col] == clusters[1]).sum()]
            ])
            odds, p_fisher = fisher_exact(ct_ward)
            print(f'Ward cluster Fisher exact: OR={odds:.3f}, p={p_fisher:.4f} ({fmt_p(p_fisher)})')
        else:
            chi2_w, p_fisher, _, _ = chi2_contingency(
                np.array([[(human_w[ward_col] == c).sum() for c in clusters],
                           [(ai_w[ward_col] == c).sum() for c in clusters]]))
            print(f'Ward cluster chi2={chi2_w:.3f}, p={p_fisher:.4f}')

        clust_rows = []
        for c in clusters:
            clust_rows.append({
                'cluster': c,
                'human_count': (human_w[ward_col] == c).sum(),
                'ai_count':    (ai_w[ward_col] == c).sum(),
                'human_frac':  (human_w[ward_col] == c).mean(),
                'ai_frac':     (ai_w[ward_col] == c).mean(),
                'fisher_p':    float(p_fisher),
            })
        clust_df = pd.DataFrame(clust_rows)
        clust_df.to_csv(TABLES_DIR_AI / 'cluster_membership_human_vs_allai.csv', index=False)

        # Copy supplementary per-model CSV
        import shutil
        _src = TABLES_DIR / 'diversity_cluster_membership_by_group.csv'
        if _src.exists():
            shutil.copy(_src, TABLES_DIR_AI / 'supplementary_cluster_membership_by_model.csv')

        cluster_title_map_13 = tc_ward.set_index('title')[ward_col].to_dict()
        cluster_aligned_13 = proposal_meta['title'].map(cluster_title_map_13)
        ward_labels_full = cluster_aligned_13.to_numpy()
        cluster_colors_13 = dict(zip(clusters, plt.cm.Set2(np.linspace(0, 1, max(len(clusters), 1)))))

        # ── Figure ──────────────────────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle('Ward Cluster Membership\nHuman vs All-AI', fontsize=13)

        # Panel A: stacked bar
        ax = axes[0]
        group_names = [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})']
        bottoms = np.zeros(2)
        for c in clusters:
            cc = cluster_colors_13[c]
            vals = np.array([
                (human_w[ward_col] == c).mean(),
                (ai_w[ward_col] == c).mean()
            ])
            ax.bar([0, 1], vals, bottom=bottoms, color=cc, label=str(c), alpha=0.8,
                   edgecolor='black', linewidth=0.5)
            bottoms += vals
        ax.set_xticks([0, 1])
        ax.set_xticklabels(group_names)
        ax.set_ylabel('Fraction of proposals')
        ax.set_title(f'Cluster composition\n(Fisher p={p_fisher:.3f})')
        ax.legend(title='Ward Cluster', bbox_to_anchor=(1.0, 1), loc='upper left', fontsize=8)
        ax.yaxis.grid(True, alpha=0.25)

        # Panel B: UMAP scatter by binary group, with the same metadata rings used elsewhere.
        ax2 = axes[1]
        for c in clusters:
            mask_c = cluster_aligned_13.eq(c).to_numpy()
            if mask_c.sum() >= 3:
                try:
                    from scipy.spatial import ConvexHull
                    pts_c = X_umap2d[mask_c]
                    hull = ConvexHull(pts_c)
                    for simplex in hull.simplices:
                        ax2.plot(pts_c[simplex, 0], pts_c[simplex, 1], color=cluster_colors_13[c],
                                 alpha=0.35, linewidth=1.2, zorder=1)
                except Exception:
                    pass
        h_ai = scatter_proposal_points(ax2, X_umap2d, np.isin(np.arange(len(proposal_meta)), ai_idx),
                                       AI_COLOR, f'All AI (n={len(ai_idx)})', marker='D',
                                       alpha=0.65, size=34, zorder=3)
        h_human = scatter_proposal_points(ax2, X_umap2d, np.isin(np.arange(len(proposal_meta)), human_idx),
                                          HUMAN_COLOR, f'Human (n={len(human_idx)})', marker='o',
                                          alpha=0.90, size=46, zorder=4)
        ax2.set_xlabel('UMAP-1')
        ax2.set_ylabel('UMAP-2')
        ax2.set_title('Proposal space (UMAP)\nGroup color + metadata rings')
        add_proposal_metadata_legend(ax2, group_handles=(h_human + h_ai), loc='best', fontsize=8)
        ax2.yaxis.grid(True, alpha=0.25)

        plt.tight_layout()
        fig.savefig(FIGURES_DIR_AI / 'cluster_membership_human_vs_allai.png',
                    dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print('Saved cluster_membership_human_vs_allai.png')



## Analysis 1.4: GMM Segregation (Human vs AI)

In [ ]:
if master_df is None:
    print('WARNING: master_df not loaded — using D_pp for distance analysis only')

# Within/between group mean distances
D_hh = D_pp[np.ix_(human_idx, human_idx)].copy()
np.fill_diagonal(D_hh, np.nan)
D_aa = D_pp[np.ix_(ai_idx, ai_idx)].copy()
np.fill_diagonal(D_aa, np.nan)
D_ha = D_pp[np.ix_(human_idx, ai_idx)]

within_human_mean = float(np.nanmean(D_hh))
within_ai_mean    = float(np.nanmean(D_aa))
between_mean      = float(np.nanmean(D_ha))
bw_ratio          = between_mean / np.mean([within_human_mean, within_ai_mean])
print(f'Within-Human mean dist: {within_human_mean:.4f}')
print(f'Within-AI mean dist:    {within_ai_mean:.4f}')
print(f'Between-group mean dist: {between_mean:.4f}')
print(f'Between/Within ratio:   {bw_ratio:.4f}')

# Permutation test for between/within ratio
rng_gmm = np.random.default_rng(45)
all_idx_arr = np.concatenate([human_idx, ai_idx])
obs_ratio = bw_ratio
perm_ratios = []
for _ in range(2000):
    perm_all = rng_gmm.permutation(all_idx_arr)
    ph = perm_all[:len(human_idx)]
    pa = perm_all[len(human_idx):]
    d_hh_p = D_pp[np.ix_(ph, ph)].copy(); np.fill_diagonal(d_hh_p, np.nan)
    d_aa_p = D_pp[np.ix_(pa, pa)].copy(); np.fill_diagonal(d_aa_p, np.nan)
    d_ha_p = D_pp[np.ix_(ph, pa)]
    w_mean_p = np.mean([np.nanmean(d_hh_p), np.nanmean(d_aa_p)])
    b_mean_p = np.nanmean(d_ha_p)
    perm_ratios.append(b_mean_p / w_mean_p if w_mean_p > 0 else np.nan)
perm_ratios = np.array([v for v in perm_ratios if np.isfinite(v)])
p_perm_bw = (np.sum(np.abs(perm_ratios - 1) >= abs(obs_ratio - 1)) + 1) / (len(perm_ratios) + 1)
print(f'B/W ratio permutation p={p_perm_bw:.4f} ({fmt_p(p_perm_bw)})')

# NMI / ARI with GMM labels (if available)
nmi_val, ari_val = np.nan, np.nan
if master_df is not None and 'remote_clique_group' in master_df.columns:
    # Use any cluster column available
    cluster_col = None
    for cc in ['gmm_cluster', 'remote_clique_group', 'ward_cluster']:
        if cc in master_df.columns:
            cluster_col = cc
            break
    if cluster_col:
        true_labels = (proposal_meta['group_binary'] == 'AI').astype(int).values
        pred_labels = master_df[cluster_col].values
        le = LabelEncoder()
        pred_enc = le.fit_transform(pred_labels)
        nmi_val = normalized_mutual_info_score(true_labels, pred_enc)
        ari_val = adjusted_rand_score(true_labels, pred_enc)
        print(f'NMI={nmi_val:.4f}, ARI={ari_val:.4f}')

gmm_seg = pd.DataFrame([{
    'within_human_mean': within_human_mean,
    'within_ai_mean': within_ai_mean,
    'between_mean': between_mean,
    'bw_ratio': bw_ratio,
    'perm_p_bw_ratio': p_perm_bw,
    'nmi': nmi_val,
    'ari': ari_val,
}])
gmm_seg.to_csv(TABLES_DIR_AI / 'gmm_segregation_human_vs_allai.csv', index=False)
print('Saved gmm_segregation_human_vs_allai.csv')


# PART II: DIVERSITY

Three-value reporting pattern for N-sensitive metrics:
- `human_metric`: human group value (n=23)
- `ai_full_metric`: all-AI value (n=69, supplementary)
- `ai_boot_mean±CI`: bootstrap-corrected (n=23-equivalent, primary)

## Analysis 2.1: Within-Group Pairwise Diversity

In [ ]:
# Per-proposal pairwise mean distances
human_pp = per_proposal_pairwise(human_idx, D_pp)
ai_full_pp = per_proposal_pairwise(ai_idx, D_pp)

# Group-level metric
human_mpd    = group_mean_pairwise(human_idx, D_pp)
ai_full_mpd  = group_mean_pairwise(ai_idx, D_pp)

# Bootstrap subsamples
boot_mpd_vals = boot_metric_over_samples(group_mean_pairwise, boot_ai_idx_samples, D_pp)
ai_boot_mpd   = float(np.mean(boot_mpd_vals))
ai_boot_mpd_ci = boot_ci(boot_mpd_vals)

# Per-model supplementary
claude_mpd = group_mean_pairwise(claude_idx, D_pp)
gemini_mpd = group_mean_pairwise(gemini_idx, D_pp)
gpt_mpd    = group_mean_pairwise(gpt_idx, D_pp)
within_model_avg = float(np.mean([claude_mpd, gemini_mpd, gpt_mpd]))

# MW test on per-proposal values (full 23 vs 69)
stat_pp, p_mw = mannwhitneyu(ai_full_pp, human_pp, alternative='two-sided')
delta    = cliffs_delta(ai_full_pp, human_pp)
d_lo, d_hi = bootstrap_cliffs_delta_ci(ai_full_pp, human_pp, n_boot=2000, random_state=42)

# Permutation test on per-proposal values (full groups)
rng_perm_pp = np.random.default_rng(42)
all_pp_vals = np.concatenate([human_pp, ai_full_pp])
obs_mean_diff = np.mean(ai_full_pp) - np.mean(human_pp)
perm_diffs_pp = []
for _ in range(10000):
    perm = rng_perm_pp.permutation(all_pp_vals)
    perm_diffs_pp.append(np.mean(perm[len(human_pp):]) - np.mean(perm[:len(human_pp)]))
p_perm_pp = (np.sum(np.abs(perm_diffs_pp) >= abs(obs_mean_diff)) + 1) / 10001

print(f'Human MPD:       {human_mpd:.4f}')
print(f'AI Full MPD:     {ai_full_mpd:.4f} (n=69)')
print(f'AI Boot MPD:     {ai_boot_mpd:.4f} [{ai_boot_mpd_ci[0]:.4f}, {ai_boot_mpd_ci[1]:.4f}] (n=23-equiv)')
print(f'Within-model avg: {within_model_avg:.4f}')
print(f"MW p={p_mw:.4f}, Perm p={p_perm_pp:.4f}, Cliff's d={delta:.3f} [{d_lo:.3f},{d_hi:.3f}]")

summary_pp = {
    'human_mpd': human_mpd, 'ai_full_mpd': ai_full_mpd,
    'ai_boot_mpd_mean': ai_boot_mpd,
    'ai_boot_ci_lo': ai_boot_mpd_ci[0], 'ai_boot_ci_hi': ai_boot_mpd_ci[1],
    'within_model_avg': within_model_avg,
    'mw_p': p_mw, 'perm_p': p_perm_pp,
    'cliffs_delta': delta, 'cliffs_delta_ci_lo': d_lo, 'cliffs_delta_ci_hi': d_hi
}
pd.DataFrame([summary_pp]).to_csv(
    TABLES_DIR_AI / 'diversity_pairwise_human_vs_allai.csv', index=False)


In [ ]:
# Analysis 2.1 figure: boxplot + forest plot
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Within-Group Pairwise Diversity\nPairwise cosine distance (per proposal mean)',
             fontsize=13)


# Panel A: boxplot with jitter + mean CI diamonds
ax_a = axes[0]
h_data = human_pp
a_data = ai_full_pp
group_data = [h_data, a_data]
group_colors_box = [HUMAN_COLOR, AI_COLOR]
xlabels = [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})']
draw_boxplot_with_jitter(
    ax_a, group_data, xlabels, group_colors_box,
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)

# Annotate bootstrap mean for AI
ax_a.axhline(ai_boot_mpd, color=AI_COLOR, linewidth=1.5, linestyle=':', alpha=0.8)
ax_a.annotate(f'Bootstrap n=23 mean\n({ai_boot_mpd:.3f})',
               xy=(1, ai_boot_mpd), xytext=(1.25, ai_boot_mpd),
               fontsize=7, color=AI_COLOR,
               arrowprops=dict(arrowstyle='->', color=AI_COLOR, lw=0.8))
ax_a.set_ylabel('Pairwise cosine distance')
ax_a.set_title(f'MW {fmt_p(p_mw)} | Perm {fmt_p(p_perm_pp)}')

# Panel B: Cliff's delta forest plot
ax_b = axes[1]
draw_effect_size_panel(ax_b, delta, d_lo, d_hi, label='All AI', p_val=p_perm_pp)
ax_b.set_title("Effect size\nCliff's δ")

plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'pairwise_diversity_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved pairwise_diversity_human_vs_allai.png')


## Analysis 2.1b: Pairwise Distance Bimodality (Decomposed)

In [ ]:
# Extract pairwise distance distributions
def _upper_tri(D, idx):
    Dg = D[np.ix_(idx, idx)]
    iu = np.triu_indices(len(idx), k=1)
    return Dg[iu]

human_pairs = _upper_tri(D_pp, human_idx)
ai_pairs_all = _upper_tri(D_pp, ai_idx)

# Within-model pairs
within_model_pairs = np.concatenate([
    _upper_tri(D_pp, claude_idx),
    _upper_tri(D_pp, gemini_idx),
    _upper_tri(D_pp, gpt_idx)
])

# Between-model pairs
between_model_pairs = []
for m1, m2 in [(claude_idx, gemini_idx), (claude_idx, gpt_idx), (gemini_idx, gpt_idx)]:
    Dcm = D_pp[np.ix_(m1, m2)]
    between_model_pairs.append(Dcm.ravel())
between_model_pairs = np.concatenate(between_model_pairs)

# Diptest (optional)
dip_results = {}
try:
    from diptest import diptest
    for name, vals in [('human', human_pairs), ('ai_all', ai_pairs_all),
                        ('within_model', within_model_pairs),
                        ('between_model', between_model_pairs)]:
        d_stat, d_p = diptest(vals)
        dip_results[name] = {'dip_stat': d_stat, 'dip_p': d_p}
        print(f'{name}: dip={d_stat:.4f}, p={d_p:.4f}')
except ImportError:
    print('diptest not installed — skipping Hartigan dip test')
    for name in ['human', 'ai_all', 'within_model', 'between_model']:
        dip_results[name] = {'dip_stat': np.nan, 'dip_p': np.nan}

bimod_df = pd.DataFrame([
    {'group': k, **v,
     'n_pairs': len([human_pairs, ai_pairs_all, within_model_pairs, between_model_pairs][i]),
     'median_dist': np.median([human_pairs, ai_pairs_all, within_model_pairs, between_model_pairs][i])}
    for i, (k, v) in enumerate(dip_results.items())
])
bimod_df.to_csv(TABLES_DIR_AI / 'diversity_pairwise_bimodality_decomposed.csv', index=False)

# Figure: 3-panel KDE
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig.suptitle('Pairwise Distance Distribution (Decomposed)\nPairwise cosine distance', fontsize=12)

ax1 = axes[0]
sns.kdeplot(human_pairs, ax=ax1, color=HUMAN_COLOR, fill=True, alpha=0.4)
ax1.axvline(np.median(human_pairs), color=HUMAN_COLOR, ls='--', lw=1.5)
ax1.set_title(f'Human (n={len(human_pairs)} pairs)')
ax1.set_xlabel('Pairwise cosine distance')
ax1.yaxis.grid(True, alpha=0.25)

ax2 = axes[1]
sns.kdeplot(within_model_pairs, ax=ax2, color='steelblue', fill=True, alpha=0.4,
            label='Within-model')
sns.kdeplot(between_model_pairs, ax=ax2, color='darkorange', fill=True, alpha=0.4,
            label='Between-model')
ax2.axvline(np.median(ai_pairs_all), color=AI_COLOR, ls='--', lw=1.5)
ax2.set_title(f'Pooled AI (n={len(ai_pairs_all)} pairs)\n'
              'Bimodality reflects model mixing')
ax2.set_xlabel('Pairwise cosine distance')
ax2.legend(fontsize=8)
ax2.yaxis.grid(True, alpha=0.25)

ax3 = axes[2]
sns.kdeplot(within_model_pairs, ax=ax3, color='steelblue', fill=True, alpha=0.5)
ax3.axvline(np.median(within_model_pairs), color='steelblue', ls='--', lw=1.5)
ax3.set_title(f'Within-model AI only\n(n={len(within_model_pairs)} pairs)')
ax3.set_xlabel('Pairwise cosine distance')
ax3.yaxis.grid(True, alpha=0.25)

plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'pairwise_diversity_bimodality_decomposed.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved pairwise_diversity_bimodality_decomposed.png')


## Analysis 2.1c: Cross-Group Topic Space Alignment

In [ ]:
ai_to_nearest_human  = np.array([np.min(D_pp[i, human_idx]) for i in ai_idx])
human_to_nearest_ai  = np.array([np.min(D_pp[j, ai_idx]) for j in human_idx])

stat_cross, p_cross = mannwhitneyu(ai_to_nearest_human, human_to_nearest_ai,
                                    alternative='two-sided')
delta_cross = cliffs_delta(ai_to_nearest_human, human_to_nearest_ai)
d_lo_c, d_hi_c = bootstrap_cliffs_delta_ci(ai_to_nearest_human, human_to_nearest_ai)

print(f'AI-to-nearest-Human: mean={np.mean(ai_to_nearest_human):.4f}')
print(f'Human-to-nearest-AI: mean={np.mean(human_to_nearest_ai):.4f}')
print(f"MW p={p_cross:.4f} ({fmt_p(p_cross)}), Cliff's d={delta_cross:.3f}")

pd.DataFrame([{
    'ai_to_nearest_human_mean': float(np.mean(ai_to_nearest_human)),
    'human_to_nearest_ai_mean': float(np.mean(human_to_nearest_ai)),
    'mw_p': p_cross, 'cliffs_delta': delta_cross,
    'cliffs_delta_ci_lo': d_lo_c, 'cliffs_delta_ci_hi': d_hi_c
}]).to_csv(TABLES_DIR_AI / 'diversity_cross_group_alignment_human_vs_allai.csv', index=False)


# Visualization: mirror rephrased Analysis 2.1c in pooled Human vs All-AI form.
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Cross-Group Topic Space Alignment\nNearest opposite-group cosine distance', fontsize=13)
draw_boxplot_with_jitter(
    axes[0],
    [human_to_nearest_ai, ai_to_nearest_human],
    [f'Human -> nearest AI\n(n={len(human_to_nearest_ai)})', f'All AI -> nearest Human\n(n={len(ai_to_nearest_human)})'],
    [HUMAN_COLOR, AI_COLOR],
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axes[0].set_ylabel('Min cosine distance to nearest opposite-group proposal')
axes[0].set_title(f'MW {fmt_p(p_cross)}')
draw_effect_size_panel(axes[1], delta_cross, d_lo_c, d_hi_c, p_val=p_cross)
axes[1].set_title("Effect size\nCliff's delta")
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'cross_group_topic_alignment_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved cross_group_topic_alignment_human_vs_allai.png')


## Analysis 2.1d: Within-Cluster Diversity

In [ ]:

if topic_cluster_df is None:
    print('WARNING: topic_cluster_df not loaded - skipping Analysis 2.1d')
else:
    tc_c = topic_cluster_df.merge(
        proposal_meta[['title', 'group_binary']], on='title', how='left')
    tc_c['group_binary'] = tc_c['group_binary'].fillna('AI')
    ward_col2 = 'ward_cluster_name' if 'ward_cluster_name' in tc_c.columns else 'ward_cluster_display'

    if ward_col2 not in tc_c.columns:
        print('WARNING: ward cluster column not found')
    else:
        wc_rows = []
        between_rows = []
        clusters_wc = sorted(tc_c[ward_col2].dropna().unique())
        group_specs_wc = [('Human', human_idx), ('All AI', ai_idx)]

        cluster_to_idx = {}
        for cname in clusters_wc:
            titles_in_cluster = tc_c[tc_c[ward_col2] == cname]['title'].values
            cluster_to_idx[cname] = proposal_meta.index[
                proposal_meta['title'].isin(titles_in_cluster)
            ].to_numpy()

        for cname in clusters_wc:
            idx_in_cluster = cluster_to_idx[cname]
            for group_name, group_idx in group_specs_wc:
                g_in = np.intersect1d(idx_in_cluster, group_idx)
                row = {
                    'cluster': cname,
                    'group': group_name,
                    'n_proposals': int(len(g_in)),
                    'mean_pairwise_dist': np.nan,
                }
                if len(g_in) >= 2:
                    sub_D = D_pp[np.ix_(g_in, g_in)]
                    upper = sub_D[np.triu_indices_from(sub_D, k=1)]
                    row['mean_pairwise_dist'] = float(np.nanmean(upper))
                elif len(g_in) == 1:
                    row['mean_pairwise_dist'] = 0.0
                wc_rows.append(row)

        for group_name, group_idx in group_specs_wc:
            for i, c1 in enumerate(clusters_wc):
                for c2 in clusters_wc[i+1:]:
                    a_idx = np.intersect1d(cluster_to_idx[c1], group_idx)
                    b_idx = np.intersect1d(cluster_to_idx[c2], group_idx)
                    if len(a_idx) and len(b_idx):
                        cross_D = D_pp[np.ix_(a_idx, b_idx)]
                        between_rows.append({
                            'group': group_name,
                            'cluster_pair': f'{c1}<->{c2}',
                            'n_a': int(len(a_idx)),
                            'n_b': int(len(b_idx)),
                            'mean_cross_dist': float(np.nanmean(cross_D)),
                        })

        wc_df = pd.DataFrame(wc_rows)
        between_df = pd.DataFrame(between_rows)
        wc_df.to_csv(TABLES_DIR_AI / 'diversity_within_cluster_human_vs_allai.csv', index=False)
        between_df.to_csv(TABLES_DIR_AI / 'diversity_between_cluster_gap_human_vs_allai.csv', index=False)
        print(wc_df.to_string())
        if len(between_df):
            print('\nBetween-cluster gap:')
            print(between_df.to_string(index=False))

        # Human-vs-All-AI tests within each cluster when both sides have enough proposals.
        test_rows_wc = []
        for cname in clusters_wc:
            idx_in_cluster = cluster_to_idx[cname]
            h_in = np.intersect1d(idx_in_cluster, human_idx)
            a_in = np.intersect1d(idx_in_cluster, ai_idx)
            if len(h_in) >= 3 and len(a_in) >= 3:
                h_pp_c = per_proposal_pairwise(h_in, D_pp)
                a_pp_c = per_proposal_pairwise(a_in, D_pp)
                _, p_wc = mannwhitneyu(a_pp_c, h_pp_c, alternative='two-sided')
                test_rows_wc.append({
                    'cluster': cname,
                    'n_human': int(len(h_in)),
                    'n_ai': int(len(a_in)),
                    'human_mpd': float(np.nanmean(h_pp_c)),
                    'ai_mpd': float(np.nanmean(a_pp_c)),
                    'mw_p': float(p_wc),
                    'cliffs_delta': cliffs_delta(a_pp_c, h_pp_c),
                })
        if test_rows_wc:
            wc_test_df = pd.DataFrame(test_rows_wc)
            _, wc_test_df['p_holm'], _, _ = multipletests(wc_test_df['mw_p'], method='holm')
            wc_test_df.to_csv(TABLES_DIR_AI / 'diversity_within_cluster_tests_human_vs_allai.csv', index=False)
            print('\nWithin-cluster Human vs All-AI tests:')
            print(wc_test_df.to_string(index=False))

        # Visualization: mirror rephrased cluster-aware comparison in pooled form.
        n_cluster_panels = min(2, len(clusters_wc))
        fig, axes = plt.subplots(1, n_cluster_panels + 1, figsize=(5.4 * (n_cluster_panels + 1), 5))
        if n_cluster_panels + 1 == 1:
            axes = [axes]
        fig.suptitle('Cluster-Controlled Diversity Comparison - Human vs All-AI', fontsize=12, fontweight='bold')
        for pi, cname in enumerate(clusters_wc[:n_cluster_panels]):
            ax = axes[pi]
            df_k = wc_df[wc_df['cluster'] == cname].copy()
            x_labels = [f"{row['group']}\n(n={int(row['n_proposals'])})" for _, row in df_k.iterrows()]
            bar_c = [HUMAN_COLOR if g == 'Human' else AI_COLOR for g in df_k['group']]
            ax.bar(range(len(df_k)), df_k['mean_pairwise_dist'], color=bar_c, alpha=0.85)
            ax.set_xticks(range(len(df_k)))
            ax.set_xticklabels(x_labels, fontsize=9)
            ax.set_title(f'Within-{str(cname)[:36]} diversity')
            ax.set_ylabel('Mean pairwise cosine distance')
            ax.tick_params(axis='x', rotation=10)
            ax.yaxis.grid(True, alpha=0.25)

        ax_gap = axes[-1]
        if len(between_df):
            df_b_grp = between_df.groupby('group')['mean_cross_dist'].mean().reindex(['Human', 'All AI']).dropna().reset_index()
            bar_c = [HUMAN_COLOR if g == 'Human' else AI_COLOR for g in df_b_grp['group']]
            y_labels = [f"{row['group']}" for _, row in df_b_grp.iterrows()]
            ax_gap.barh(y_labels, df_b_grp['mean_cross_dist'], color=bar_c, alpha=0.85)
            human_gap = df_b_grp.loc[df_b_grp['group'] == 'Human', 'mean_cross_dist'].to_numpy()
            if len(human_gap):
                ax_gap.axvline(human_gap[0], ls='--', color='black', lw=1.5, label='Human baseline')
                ax_gap.legend(fontsize=8)
        ax_gap.set_title('Between-cluster gap')
        ax_gap.set_xlabel('Mean cross-cluster distance')
        ax_gap.xaxis.grid(True, alpha=0.25)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR_AI / 'diversity_cluster_aware_comparison_human_vs_allai.png',
                    dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print('Saved diversity_cluster_aware_comparison_human_vs_allai.png')


## Analysis 2.2a: Centroid Dispersion (LOO)

In [ ]:
# Per-proposal LOO centroid distances
human_loo    = per_proposal_loo_centroid(human_idx, X_prop)
ai_full_loo  = per_proposal_loo_centroid(ai_idx, X_prop)

human_loo_mean   = float(np.mean(human_loo))
ai_full_loo_mean = float(np.mean(ai_full_loo))

boot_loo_vals = boot_metric_over_samples(
    lambda s, X: float(np.mean(group_loo_centroid(s, X))),
    boot_ai_idx_samples, X_prop
)
ai_boot_loo_mean = float(np.mean(boot_loo_vals))
ai_boot_loo_ci   = boot_ci(boot_loo_vals)

human_span90       = float(np.percentile(human_loo, 90))
boot_span90_vals   = boot_metric_over_samples(
    lambda s, X: float(np.percentile(group_loo_centroid(s, X), 90)),
    boot_ai_idx_samples, X_prop
)
ai_boot_span90_mean = float(np.mean(boot_span90_vals))

stat_loo, p_mw_loo = mannwhitneyu(ai_full_loo, human_loo, alternative='two-sided')
delta_loo = cliffs_delta(ai_full_loo, human_loo)
d_lo_loo, d_hi_loo = bootstrap_cliffs_delta_ci(ai_full_loo, human_loo, n_boot=2000, random_state=42)

rng_perm_loo = np.random.default_rng(43)
all_loo_vals = np.concatenate([human_loo, ai_full_loo])
obs_diff_loo = np.mean(ai_full_loo) - np.mean(human_loo)
perm_diffs_loo = []
for _ in range(10000):
    p_perm_arr = rng_perm_loo.permutation(all_loo_vals)
    perm_diffs_loo.append(np.mean(p_perm_arr[len(human_loo):]) - np.mean(p_perm_arr[:len(human_loo)]))
p_perm_loo = (np.sum(np.abs(perm_diffs_loo) >= abs(obs_diff_loo)) + 1) / 10001

print(f'Human LOO mean:     {human_loo_mean:.4f}')
print(f'AI Full LOO mean:   {ai_full_loo_mean:.4f} (n=69)')
print(f'AI Boot LOO mean:   {ai_boot_loo_mean:.4f} [{ai_boot_loo_ci[0]:.4f}, {ai_boot_loo_ci[1]:.4f}]')
print(f"MW p={p_mw_loo:.4f}, Perm p={p_perm_loo:.4f}, Cliff's d={delta_loo:.3f}")

pd.DataFrame([{
    'human_loo_mean': human_loo_mean, 'ai_full_loo_mean': ai_full_loo_mean,
    'ai_boot_loo_mean': ai_boot_loo_mean,
    'ai_boot_loo_ci_lo': ai_boot_loo_ci[0], 'ai_boot_loo_ci_hi': ai_boot_loo_ci[1],
    'human_span90': human_span90, 'ai_boot_span90_mean': ai_boot_span90_mean,
    'mw_p': p_mw_loo, 'perm_p': p_perm_loo,
    'cliffs_delta': delta_loo, 'cliffs_delta_ci_lo': d_lo_loo, 'cliffs_delta_ci_hi': d_hi_loo
}]).to_csv(TABLES_DIR_AI / 'diversity_centroid_human_vs_allai.csv', index=False)

# Figure
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Centroid Dispersion (LOO)\nPer-proposal LOO centroid cosine distance', fontsize=13)
draw_boxplot_with_jitter(
    axes[0],
    [human_loo, ai_full_loo],
    [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})'],
    [HUMAN_COLOR, AI_COLOR],
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axes[0].axhline(ai_boot_loo_mean, color=AI_COLOR, linewidth=1.5, linestyle=':', alpha=0.8)
axes[0].set_ylabel('LOO centroid cosine distance')
axes[0].set_title(f'MW {fmt_p(p_mw_loo)} | Perm {fmt_p(p_perm_loo)}')
draw_effect_size_panel(axes[1], delta_loo, d_lo_loo, d_hi_loo, p_val=p_perm_loo)
axes[1].set_title("Effect size\nCliff's δ")
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'centroid_dispersion_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved centroid_dispersion_human_vs_allai.png')


## Analysis 2.2b: Global-Centroid Distance

In [ ]:
human_gc   = group_global_centroid(human_idx, X_prop)
ai_full_gc = group_global_centroid(ai_idx, X_prop)

boot_gc_vals   = boot_metric_over_samples(
    lambda s, X: float(np.mean(group_global_centroid(s, X))),
    boot_ai_idx_samples, X_prop
)
ai_boot_gc_mean = float(np.mean(boot_gc_vals))
ai_boot_gc_ci   = boot_ci(boot_gc_vals)

stat_gc, p_mw_gc = mannwhitneyu(ai_full_gc, human_gc, alternative='two-sided')
delta_gc = cliffs_delta(ai_full_gc, human_gc)
d_lo_gc, d_hi_gc = bootstrap_cliffs_delta_ci(ai_full_gc, human_gc)

rng_gc = np.random.default_rng(46)
all_gc = np.concatenate([human_gc, ai_full_gc])
obs_gc = np.mean(ai_full_gc) - np.mean(human_gc)
perm_gc = []
for _ in range(10000):
    p_arr = rng_gc.permutation(all_gc)
    perm_gc.append(np.mean(p_arr[len(human_gc):]) - np.mean(p_arr[:len(human_gc)]))
p_perm_gc = (np.sum(np.abs(perm_gc) >= abs(obs_gc)) + 1) / 10001

print(f'Human global centroid mean:     {np.mean(human_gc):.4f}')
print(f'AI Full global centroid mean:   {np.mean(ai_full_gc):.4f}')
print(f'AI Boot mean: {ai_boot_gc_mean:.4f} [{ai_boot_gc_ci[0]:.4f}, {ai_boot_gc_ci[1]:.4f}]')
print(f"MW p={p_mw_gc:.4f} ({fmt_p(p_mw_gc)}), Perm p={p_perm_gc:.4f} ({fmt_p(p_perm_gc)})")

pd.DataFrame([{
    'human_gc_mean': float(np.mean(human_gc)), 'ai_full_gc_mean': float(np.mean(ai_full_gc)),
    'ai_boot_gc_mean': ai_boot_gc_mean,
    'ai_boot_gc_ci_lo': ai_boot_gc_ci[0], 'ai_boot_gc_ci_hi': ai_boot_gc_ci[1],
    'mw_p': p_mw_gc, 'perm_p': p_perm_gc,
    'cliffs_delta': delta_gc, 'cliffs_delta_ci_lo': d_lo_gc, 'cliffs_delta_ci_hi': d_hi_gc
}]).to_csv(TABLES_DIR_AI / 'diversity_global_centroid_human_vs_allai.csv', index=False)


# Visualization: mirror rephrased Analysis 2.2b with Human vs All-AI pooling.
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Global-Centroid Dispersion\nPer-proposal cosine distance to global centroid', fontsize=13)
draw_boxplot_with_jitter(
    axes[0],
    [human_gc, ai_full_gc],
    [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})'],
    [HUMAN_COLOR, AI_COLOR],
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axes[0].axhline(ai_boot_gc_mean, color=AI_COLOR, linewidth=1.5, linestyle=':', alpha=0.8)
axes[0].set_ylabel('Cosine distance to global centroid')
axes[0].set_title(f'MW {fmt_p(p_mw_gc)} | Perm {fmt_p(p_perm_gc)}')
draw_effect_size_panel(axes[1], delta_gc, d_lo_gc, d_hi_gc, p_val=p_perm_gc)
axes[1].set_title("Effect size\nCliff's delta")
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'global_centroid_dispersion_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved global_centroid_dispersion_human_vs_allai.png')


## Analysis 2.2c: MST Dispersion

In [ ]:
human_mst   = group_mst_dispersion(human_idx, D_pp)
ai_full_mst = group_mst_dispersion(ai_idx, D_pp)

boot_mst_vals   = boot_metric_over_samples(group_mst_dispersion, boot_ai_idx_samples, D_pp)
ai_boot_mst_mean = float(np.mean(boot_mst_vals))
ai_boot_mst_ci   = boot_ci(boot_mst_vals)

# Permutation test (group-level, permute binary label)
rng_mst = np.random.default_rng(44)
all_idx_mst = np.concatenate([human_idx, ai_idx])
obs_mst_diff = ai_full_mst - human_mst
perm_mst_diffs = []
for _ in range(5000):
    perm = rng_mst.permutation(all_idx_mst)
    h_perm = perm[:len(human_idx)]
    a_perm = perm[len(human_idx):]
    perm_mst_diffs.append(
        group_mst_dispersion(a_perm, D_pp) - group_mst_dispersion(h_perm, D_pp))
perm_mst_diffs = np.array(perm_mst_diffs)
p_perm_mst = (np.sum(np.abs(perm_mst_diffs) >= abs(obs_mst_diff)) + 1) / 5001

print(f'Human MST mean edge: {human_mst:.4f}')
print(f'AI Full MST mean edge: {ai_full_mst:.4f} (n=69)')
print(f'AI Boot MST mean: {ai_boot_mst_mean:.4f} [{ai_boot_mst_ci[0]:.4f}, {ai_boot_mst_ci[1]:.4f}]')
print(f'Observed diff={obs_mst_diff:.4f}, Perm p={p_perm_mst:.4f} ({fmt_p(p_perm_mst)})')

pd.DataFrame([{
    'human_mst': human_mst, 'ai_full_mst': ai_full_mst,
    'ai_boot_mst_mean': ai_boot_mst_mean,
    'ai_boot_mst_ci_lo': ai_boot_mst_ci[0], 'ai_boot_mst_ci_hi': ai_boot_mst_ci[1],
    'obs_diff': obs_mst_diff, 'perm_p': p_perm_mst
}]).to_csv(TABLES_DIR_AI / 'diversity_mst_human_vs_allai.csv', index=False)

# Figure: bar chart
fig, ax = plt.subplots(figsize=(6, 5))
fig.suptitle('MST Dispersion\nMST mean edge weight (avg distance per proposal)', fontsize=12)
bar_vals = [human_mst, ai_boot_mst_mean]
bar_errs = [0, (ai_boot_mst_ci[1] - ai_boot_mst_ci[0]) / 2]
bars = ax.bar([0, 1], bar_vals, color=[HUMAN_COLOR, AI_COLOR], alpha=0.75, width=0.5)
ax.errorbar(1, ai_boot_mst_mean,
            yerr=[[ai_boot_mst_mean - ai_boot_mst_ci[0]],
                  [ai_boot_mst_ci[1] - ai_boot_mst_mean]],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})\n[bootstrap n=23]'])
ax.set_ylabel('MST mean edge weight (cosine)')
ax.set_title(f'Perm {fmt_p(p_perm_mst)}')
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'diversity_mst_human_vs_allai.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved diversity_mst_human_vs_allai.png')


## Analysis 2.2d: Sparseness (Medoid-Based Dispersion)

In [ ]:
human_spar   = group_sparseness(human_idx, D_pp)
ai_full_spar = group_sparseness(ai_idx, D_pp)

boot_spar_vals   = boot_metric_over_samples(group_sparseness, boot_ai_idx_samples, D_pp)
ai_boot_spar_mean = float(np.mean(boot_spar_vals))
ai_boot_spar_ci   = boot_ci(boot_spar_vals)

# Per-proposal medoid distances for MW
human_med_d  = per_proposal_medoid_dist(human_idx, D_pp)
ai_full_med_d = per_proposal_medoid_dist(ai_idx, D_pp)
stat_spar, p_mw_spar = mannwhitneyu(ai_full_med_d, human_med_d, alternative='two-sided')
delta_spar = cliffs_delta(ai_full_med_d, human_med_d)
d_lo_spar, d_hi_spar = bootstrap_cliffs_delta_ci(ai_full_med_d, human_med_d)

# Permutation on per-proposal values
rng_spar = np.random.default_rng(47)
all_spar = np.concatenate([human_med_d, ai_full_med_d])
obs_spar = np.mean(ai_full_med_d) - np.mean(human_med_d)
perm_spar = []
for _ in range(10000):
    p_arr = rng_spar.permutation(all_spar)
    perm_spar.append(np.mean(p_arr[len(human_med_d):]) - np.mean(p_arr[:len(human_med_d)]))
p_perm_spar = (np.sum(np.abs(perm_spar) >= abs(obs_spar)) + 1) / 10001

print(f'Human sparseness: {human_spar:.4f}')
print(f'AI Full sparseness: {ai_full_spar:.4f}')
print(f'AI Boot: {ai_boot_spar_mean:.4f} [{ai_boot_spar_ci[0]:.4f}, {ai_boot_spar_ci[1]:.4f}]')
print(f"MW p={p_mw_spar:.4f}, Perm p={p_perm_spar:.4f}, Cliff's d={delta_spar:.3f}")

pd.DataFrame([{
    'human_sparseness': human_spar, 'ai_full_sparseness': ai_full_spar,
    'ai_boot_spar_mean': ai_boot_spar_mean,
    'ai_boot_spar_ci_lo': ai_boot_spar_ci[0], 'ai_boot_spar_ci_hi': ai_boot_spar_ci[1],
    'mw_p': p_mw_spar, 'perm_p': p_perm_spar,
    'cliffs_delta': delta_spar, 'cliffs_delta_ci_lo': d_lo_spar, 'cliffs_delta_ci_hi': d_hi_spar
}]).to_csv(TABLES_DIR_AI / 'diversity_sparseness_human_vs_allai.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Sparseness (Medoid-Based Dispersion)\nPer-proposal distance to group medoid', fontsize=13)
draw_boxplot_with_jitter(
    axes[0],
    [human_med_d, ai_full_med_d],
    [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})'],
    [HUMAN_COLOR, AI_COLOR],
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axes[0].axhline(ai_boot_spar_mean, color=AI_COLOR, linewidth=1.5, linestyle=':', alpha=0.8)
axes[0].set_ylabel('Distance to group medoid (cosine)')
axes[0].set_title(f'MW {fmt_p(p_mw_spar)} | Perm {fmt_p(p_perm_spar)}')
draw_effect_size_panel(axes[1], delta_spar, d_lo_spar, d_hi_spar, p_val=p_perm_spar)
axes[1].set_title("Effect size\nCliff's δ")
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'diversity_sparseness_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved diversity_sparseness_human_vs_allai.png')


## Analysis 2.3: Nearest-Neighbor Isolation (Chamfer)

In [ ]:
# Per-proposal NN distances (within-group)
human_nn   = per_proposal_nn(human_idx, D_pp)
ai_full_nn = per_proposal_nn(ai_idx, D_pp)

# Group-level Chamfer mean
human_chamfer   = group_chamfer(human_idx, D_pp)
ai_full_chamfer = group_chamfer(ai_idx, D_pp)
boot_cham_vals   = boot_metric_over_samples(group_chamfer, boot_ai_idx_samples, D_pp)
ai_boot_cham_mean = float(np.mean(boot_cham_vals))
ai_boot_cham_ci   = boot_ci(boot_cham_vals)

stat_nn, p_mw_nn = mannwhitneyu(ai_full_nn, human_nn, alternative='two-sided')
delta_nn = cliffs_delta(ai_full_nn, human_nn)
d_lo_nn, d_hi_nn = bootstrap_cliffs_delta_ci(ai_full_nn, human_nn)

rng_nn = np.random.default_rng(48)
all_nn = np.concatenate([human_nn, ai_full_nn])
obs_nn = np.mean(ai_full_nn) - np.mean(human_nn)
perm_nn = []
for _ in range(10000):
    p_arr = rng_nn.permutation(all_nn)
    perm_nn.append(np.mean(p_arr[len(human_nn):]) - np.mean(p_arr[:len(human_nn)]))
p_perm_nn = (np.sum(np.abs(perm_nn) >= abs(obs_nn)) + 1) / 10001

# NN source composition (global nearest neighbor excl. self)
nn_source_rows = []
for i in range(len(D_pp)):
    nn_j = int(np.argmin(D_pp_infdiag[i]))
    if i in human_idx:
        src_group = 'Human'
        src = 'other_human'
    else:
        src_group = 'AI'
        # find model of nn_j
        if nn_j in human_idx:
            src = 'human'
        elif nn_j in claude_idx:
            src = 'claude'
        elif nn_j in gemini_idx:
            src = 'gemini'
        elif nn_j in gpt_idx:
            src = 'gpt'
        else:
            src = 'unknown'
    nn_source_rows.append({'proposal_idx': i, 'group': src_group,
                            'nn_idx': nn_j, 'nn_source': src,
                            'nn_dist': float(D_pp_infdiag[i, nn_j])})
nn_source_df = pd.DataFrame(nn_source_rows)
nn_source_df.to_csv(TABLES_DIR_AI / 'nearest_neighbor_source_composition_human_vs_allai.csv',
                     index=False)

print(f'Human Chamfer: {human_chamfer:.4f}')
print(f'AI Full Chamfer: {ai_full_chamfer:.4f}')
print(f'AI Boot Chamfer: {ai_boot_cham_mean:.4f} [{ai_boot_cham_ci[0]:.4f}, {ai_boot_cham_ci[1]:.4f}]')
print(f"MW p={p_mw_nn:.4f} ({fmt_p(p_mw_nn)}), Perm p={p_perm_nn:.4f} ({fmt_p(p_perm_nn)})")
print('\nNN source composition (AI proposals):')
print(nn_source_df[nn_source_df['group'] == 'AI']['nn_source'].value_counts())

pd.DataFrame([{
    'human_chamfer': human_chamfer, 'ai_full_chamfer': ai_full_chamfer,
    'ai_boot_cham_mean': ai_boot_cham_mean,
    'ai_boot_cham_ci_lo': ai_boot_cham_ci[0], 'ai_boot_cham_ci_hi': ai_boot_cham_ci[1],
    'mw_p': p_mw_nn, 'perm_p': p_perm_nn,
    'cliffs_delta': delta_nn, 'cliffs_delta_ci_lo': d_lo_nn, 'cliffs_delta_ci_hi': d_hi_nn
}]).to_csv(TABLES_DIR_AI / 'diversity_chamfer_human_vs_allai.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Nearest-Neighbor Isolation (Chamfer)\nWithin-group NN cosine distance', fontsize=13)
draw_boxplot_with_jitter(
    axes[0],
    [human_nn, ai_full_nn],
    [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})'],
    [HUMAN_COLOR, AI_COLOR],
    point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axes[0].axhline(ai_boot_cham_mean, color=AI_COLOR, linewidth=1.5, linestyle=':', alpha=0.8)
axes[0].set_ylabel('Within-group NN cosine distance')
axes[0].set_title(f'MW {fmt_p(p_mw_nn)} | Perm {fmt_p(p_perm_nn)}')
draw_effect_size_panel(axes[1], delta_nn, d_lo_nn, d_hi_nn, p_val=p_perm_nn)
axes[1].set_title("Effect size\nCliff's δ")
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'nearest_neighbor_human_vs_allai.png',
            dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved nearest_neighbor_human_vs_allai.png')


## Analysis 2.4: UMAP Embedding Space Visualization

In [ ]:
print('='*85)
print('ANALYSIS 2.4: PROPOSAL EMBEDDING SPACE UMAP - HUMAN VS ALL-AI')
print('='*85)

_top5_umap_24 = proposal_meta.get('is_top5_ranked', pd.Series(False, index=proposal_meta.index)).fillna(False).map(_coerce_bool).to_numpy(dtype=bool)
_funding_umap_24 = pd.to_numeric(proposal_meta.get('funding', pd.Series(np.nan, index=proposal_meta.index)), errors='coerce').to_numpy()
_human_mask_24 = proposal_meta['group_binary'].eq('Human').to_numpy()
_ai_mask_24 = proposal_meta['group_binary'].eq('AI').to_numpy()

# Align Ward cluster labels to proposal_meta rows and keep the richer display labels from the rephrased notebook.
_cluster_aligned_24 = pd.Series(index=proposal_meta.index, dtype=object)
_cluster_display_map_24 = {}
if topic_cluster_df is not None:
    _ward_col_24 = 'ward_cluster_name' if 'ward_cluster_name' in topic_cluster_df.columns else 'ward_cluster_display'
    if _ward_col_24 in topic_cluster_df.columns:
        _cluster_aligned_24 = proposal_meta['title'].map(topic_cluster_df.set_index('title')[_ward_col_24].to_dict())
        if 'ward_cluster_display' in topic_cluster_df.columns:
            _cluster_display_map_24 = (topic_cluster_df.dropna(subset=[_ward_col_24])
                                      .drop_duplicates(_ward_col_24)
                                      .set_index(_ward_col_24)['ward_cluster_display'].astype(str).to_dict())
        else:
            _cluster_display_map_24 = {c: str(c) for c in _cluster_aligned_24.dropna().unique()}
else:
    print('WARNING: topic_cluster_df not loaded; drawing UMAP without Ward cluster hulls/zoom labels.')

_clusters_24 = sorted([c for c in _cluster_aligned_24.dropna().unique()])
_cluster_colors_24 = dict(zip(_clusters_24, plt.cm.Set2(np.linspace(0, 1, max(len(_clusters_24), 1)))))

fig, ax = plt.subplots(figsize=(9.5, 7.5))
fig.suptitle('Proposal Embedding Space (UMAP)\nHuman vs All-AI group coloring with Ward cluster regions', fontsize=13)

# Ward cluster convex hulls and labels mirror the rephrased cluster-detail view.
if _clusters_24:
    try:
        from scipy.spatial import ConvexHull
        for cname in _clusters_24:
            mask_c = _cluster_aligned_24.eq(cname).to_numpy()
            pts_c = X_umap2d[mask_c]
            if len(pts_c) >= 3:
                hull = ConvexHull(pts_c)
                for simplex in hull.simplices:
                    ax.plot(pts_c[simplex, 0], pts_c[simplex, 1],
                            color=_cluster_colors_24[cname], alpha=0.34, linewidth=1.2, zorder=1)
                label = _cluster_display_map_24.get(cname, str(cname))
                ax.text(pts_c[:, 0].mean(), pts_c[:, 1].mean(), textwrap.fill(label, width=30),
                        fontsize=7.5, ha='center', va='center', alpha=0.78,
                        bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white',
                              'edgecolor': _cluster_colors_24[cname], 'alpha': 0.82}, zorder=2)
    except Exception as exc:
        print(f'WARNING: could not draw Ward cluster hulls ({exc})')

h_ai_24 = scatter_proposal_points(ax, X_umap2d, _ai_mask_24, AI_COLOR, f'All AI (n={int(_ai_mask_24.sum())})',
                                  marker='D', alpha=0.62, size=36, zorder=3)
h_human_24 = scatter_proposal_points(ax, X_umap2d, _human_mask_24, HUMAN_COLOR, f'Human (n={int(_human_mask_24.sum())})',
                                     marker='o', alpha=0.92, size=54, zorder=4)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.yaxis.grid(True, alpha=0.25)
add_proposal_metadata_legend(ax, group_handles=(h_human_24 + h_ai_24), loc='best', fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'embedding_space_umap_human_vs_allai.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved embedding_space_umap_human_vs_allai.png')

# Per-cluster zoom panels: same group colors/rings, richer topic labels, and local bounding boxes.
if _clusters_24:
    _n_panels_24 = min(len(_clusters_24), 4)
    fig2, axes2 = plt.subplots(1, _n_panels_24, figsize=(7 * _n_panels_24, 6.6), squeeze=False)
    axes2 = axes2[0]
    for cname, axz in zip(_clusters_24[:_n_panels_24], axes2):
        mask_c = _cluster_aligned_24.eq(cname).to_numpy()
        pts_c = X_umap2d[mask_c]
        h_mask_c = mask_c & _human_mask_24
        a_mask_c = mask_c & _ai_mask_24

        axz.scatter(X_umap2d[~mask_c, 0], X_umap2d[~mask_c, 1], c='#d0d0d0', s=10,
                    alpha=0.18, edgecolors='none', zorder=1)
        h_ai_z = scatter_proposal_points(axz, X_umap2d, a_mask_c, AI_COLOR, f'All AI (n={int(a_mask_c.sum())})',
                                         marker='D', alpha=0.70, size=50, zorder=3)
        h_hu_z = scatter_proposal_points(axz, X_umap2d, h_mask_c, HUMAN_COLOR, f'Human (n={int(h_mask_c.sum())})',
                                         marker='o', alpha=0.95, size=68, zorder=4)

        if len(pts_c):
            px = max(float(np.ptp(pts_c[:, 0])) * 0.25, 0.45)
            py = max(float(np.ptp(pts_c[:, 1])) * 0.25, 0.45)
            axz.set_xlim(float(pts_c[:, 0].min()) - px, float(pts_c[:, 0].max()) + px)
            axz.set_ylim(float(pts_c[:, 1].min()) - py, float(pts_c[:, 1].max()) + py)

        label = _cluster_display_map_24.get(cname, str(cname))
        axz.set_title(f'{label}\nHuman n={int(h_mask_c.sum())}; All AI n={int(a_mask_c.sum())}; total n={int(mask_c.sum())}',
                      fontsize=10.5, fontweight='bold')
        axz.set_xlabel('UMAP-1')
        axz.set_ylabel('UMAP-2')
        axz.grid(alpha=0.2, linestyle='--')
        add_proposal_metadata_legend(axz, group_handles=(h_hu_z + h_ai_z), loc='best', fontsize=8)

    fig2.suptitle('Per-Cluster Zoom UMAP Projection for All Proposals\nmagenta ring = funded Human; black ring = top-ranked proposal',
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig2.savefig(FIGURES_DIR_AI / 'embedding_space_umap_per_cluster_zoom_human_vs_allai.png',
                 dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig2)
    print('Saved embedding_space_umap_per_cluster_zoom_human_vs_allai.png')
else:
    print('WARNING: no Ward cluster labels available; skipped per-cluster UMAP zoom.')



## Analysis 2.5: Grid Entropy

In [ ]:
human_ent   = group_grid_entropy(human_idx, X_pca2d)
ai_full_ent = group_grid_entropy(ai_idx, X_pca2d)

boot_ent_vals   = boot_metric_over_samples(group_grid_entropy, boot_ai_idx_samples, X_pca2d)
ai_boot_ent_mean = float(np.mean(boot_ent_vals))
ai_boot_ent_ci   = boot_ci(boot_ent_vals)

# Group-level permutation (permute binary assignment)
rng_ent = np.random.default_rng(49)
all_idx_ent = np.concatenate([human_idx, ai_idx])
obs_ent_diff = ai_full_ent - human_ent
perm_ent_diffs = []
for _ in range(5000):
    perm = rng_ent.permutation(all_idx_ent)
    h_p = perm[:len(human_idx)]
    a_p = perm[len(human_idx):]
    perm_ent_diffs.append(
        group_grid_entropy(a_p, X_pca2d) - group_grid_entropy(h_p, X_pca2d))
perm_ent_diffs = np.array(perm_ent_diffs)
p_perm_ent = (np.sum(np.abs(perm_ent_diffs) >= abs(obs_ent_diff)) + 1) / 5001

print(f'Human entropy (norm): {human_ent:.4f}')
print(f'AI Full entropy:      {ai_full_ent:.4f} (n=69)')
print(f'AI Boot entropy mean: {ai_boot_ent_mean:.4f} [{ai_boot_ent_ci[0]:.4f}, {ai_boot_ent_ci[1]:.4f}]')
print(f'Perm p={p_perm_ent:.4f} ({fmt_p(p_perm_ent)})')

pd.DataFrame([{
    'human_entropy': human_ent, 'ai_full_entropy': ai_full_ent,
    'ai_boot_ent_mean': ai_boot_ent_mean,
    'ai_boot_ent_ci_lo': ai_boot_ent_ci[0], 'ai_boot_ent_ci_hi': ai_boot_ent_ci[1],
    'obs_diff': obs_ent_diff, 'perm_p': p_perm_ent
}]).to_csv(TABLES_DIR_AI / 'diversity_entropy_human_vs_allai.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 5))
fig.suptitle('Grid Entropy\nNormalized 5x5-grid entropy of PCA-2D layout', fontsize=12)
bar_vals_e = [human_ent, ai_boot_ent_mean]
ax.bar([0, 1], bar_vals_e, color=[HUMAN_COLOR, AI_COLOR], alpha=0.75, width=0.5)
ax.errorbar(1, ai_boot_ent_mean,
            yerr=[[ai_boot_ent_mean - ai_boot_ent_ci[0]],
                  [ai_boot_ent_ci[1] - ai_boot_ent_mean]],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})\n[bootstrap n=23]'])
ax.set_ylabel('Normalized grid entropy')
ax.set_title(f'Perm {fmt_p(p_perm_ent)}')
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(FIGURES_DIR_AI / 'diversity_entropy_human_vs_allai.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved diversity_entropy_human_vs_allai.png')


## Part II Diversity Summary Table

In [ ]:
div_summary_rows = [
    {'metric': 'Mean Pairwise Distance',
     'human_val': human_mpd, 'ai_full_val': ai_full_mpd,
     'ai_boot_mean': ai_boot_mpd, 'ai_boot_ci_lo': ai_boot_mpd_ci[0], 'ai_boot_ci_hi': ai_boot_mpd_ci[1],
     'mw_p': p_mw, 'perm_p': p_perm_pp, 'cliffs_delta': delta},
    {'metric': 'LOO Centroid Distance',
     'human_val': human_loo_mean, 'ai_full_val': ai_full_loo_mean,
     'ai_boot_mean': ai_boot_loo_mean, 'ai_boot_ci_lo': ai_boot_loo_ci[0], 'ai_boot_ci_hi': ai_boot_loo_ci[1],
     'mw_p': p_mw_loo, 'perm_p': p_perm_loo, 'cliffs_delta': delta_loo},
    {'metric': 'Global Centroid Distance',
     'human_val': float(np.mean(human_gc)), 'ai_full_val': float(np.mean(ai_full_gc)),
     'ai_boot_mean': ai_boot_gc_mean, 'ai_boot_ci_lo': ai_boot_gc_ci[0], 'ai_boot_ci_hi': ai_boot_gc_ci[1],
     'mw_p': p_mw_gc, 'perm_p': p_perm_gc, 'cliffs_delta': delta_gc},
    {'metric': 'MST Mean Edge Weight',
     'human_val': human_mst, 'ai_full_val': ai_full_mst,
     'ai_boot_mean': ai_boot_mst_mean, 'ai_boot_ci_lo': ai_boot_mst_ci[0], 'ai_boot_ci_hi': ai_boot_mst_ci[1],
     'mw_p': np.nan, 'perm_p': p_perm_mst, 'cliffs_delta': np.nan},
    {'metric': 'Sparseness (Medoid)',
     'human_val': human_spar, 'ai_full_val': ai_full_spar,
     'ai_boot_mean': ai_boot_spar_mean, 'ai_boot_ci_lo': ai_boot_spar_ci[0], 'ai_boot_ci_hi': ai_boot_spar_ci[1],
     'mw_p': p_mw_spar, 'perm_p': p_perm_spar, 'cliffs_delta': delta_spar},
    {'metric': 'Chamfer (NN Distance)',
     'human_val': human_chamfer, 'ai_full_val': ai_full_chamfer,
     'ai_boot_mean': ai_boot_cham_mean, 'ai_boot_ci_lo': ai_boot_cham_ci[0], 'ai_boot_ci_hi': ai_boot_cham_ci[1],
     'mw_p': p_mw_nn, 'perm_p': p_perm_nn, 'cliffs_delta': delta_nn},
    {'metric': 'Grid Entropy (norm)',
     'human_val': human_ent, 'ai_full_val': ai_full_ent,
     'ai_boot_mean': ai_boot_ent_mean, 'ai_boot_ci_lo': ai_boot_ent_ci[0], 'ai_boot_ci_hi': ai_boot_ent_ci[1],
     'mw_p': np.nan, 'perm_p': p_perm_ent, 'cliffs_delta': np.nan},
]
div_summary_df = pd.DataFrame(div_summary_rows)
div_summary_df['mw_sig'] = div_summary_df['mw_p'].apply(fmt_p)
div_summary_df['perm_sig'] = div_summary_df['perm_p'].apply(fmt_p)
div_summary_df.to_csv(TABLES_DIR_AI / 'diversity_summary_human_vs_allai.csv', index=False)
print('Diversity Summary:')
print(div_summary_df[['metric', 'human_val', 'ai_boot_mean', 'perm_p', 'perm_sig', 'cliffs_delta']].to_string())


# PART III: NOVELTY

All novelty metrics are per-proposal. Tested with Mann-Whitney U. No subsampling needed.
All cached CSVs were loaded at the top.

## Analysis 3.2.5: Element Novelty Percentiles

In [ ]:
if novelty_element_df is None:
    print('WARNING: novelty_element_df not loaded — skipping Analysis 3.2.5')
else:
    el_cols = [c for c in novelty_element_df.columns if c.startswith('element_novel')]
    human_el = novelty_element_df[novelty_element_df['group_binary'] == 'Human']
    ai_el    = novelty_element_df[novelty_element_df['group_binary'] == 'AI']
    el_rows = []
    for col in el_cols:
        h_v = human_el[col].values
        a_v = ai_el[col].values
        stat_el, p_el = mannwhitneyu(a_v, h_v, alternative='two-sided')
        d_el = cliffs_delta(a_v, h_v)
        d_lo_el, d_hi_el = bootstrap_cliffs_delta_ci(a_v, h_v)
        el_rows.append({
            'metric': col,
            'human_mean': float(np.mean(h_v)), 'ai_mean': float(np.mean(a_v)),
            'mw_p': p_el, 'cliffs_delta': d_el,
            'cliffs_delta_ci_lo': d_lo_el, 'cliffs_delta_ci_hi': d_hi_el
        })
    el_df = pd.DataFrame(el_rows)
    if len(el_df) > 0:
        _, el_df['p_holm'], _, _ = multipletests(el_df['mw_p'], method='holm')
    el_df['mw_sig'] = el_df['mw_p'].apply(fmt_p)
    el_df.to_csv(TABLES_DIR_AI / 'novelty_element_percentiles_human_vs_allai.csv', index=False)
    print(el_df[['metric', 'human_mean', 'ai_mean', 'mw_p', 'p_holm', 'cliffs_delta']].to_string())


## Step 3: Mean kNN Novelty Scores and Local Density

In [ ]:
knn_rows = []
if novelty_knn_df is not None:
    knn_cols = [c for c in novelty_knn_df.columns if c.startswith('mean_knn')]
    human_knn = novelty_knn_df[novelty_knn_df['group_binary'] == 'Human']
    ai_knn    = novelty_knn_df[novelty_knn_df['group_binary'] == 'AI']
    for col in knn_cols:
        h_v = human_knn[col].values
        a_v = ai_knn[col].values
        stat_k, p_k = mannwhitneyu(a_v, h_v, alternative='two-sided')
        d_k = cliffs_delta(a_v, h_v)
        d_lo_k, d_hi_k = bootstrap_cliffs_delta_ci(a_v, h_v)
        knn_rows.append({
            'metric': col, 'human_mean': float(np.mean(h_v)), 'ai_mean': float(np.mean(a_v)),
            'mw_p': p_k, 'cliffs_delta': d_k,
            'cliffs_delta_ci_lo': d_lo_k, 'cliffs_delta_ci_hi': d_hi_k
        })

if novelty_norm_df is not None:
    norm_cols = [c for c in ['novelty_ratio', 'novelty_z'] if c in novelty_norm_df.columns]
    human_norm = novelty_norm_df[novelty_norm_df['group_binary'] == 'Human']
    ai_norm    = novelty_norm_df[novelty_norm_df['group_binary'] == 'AI']
    for col in norm_cols:
        h_v = human_norm[col].values
        a_v = ai_norm[col].values
        stat_n, p_n = mannwhitneyu(a_v, h_v, alternative='two-sided')
        d_n = cliffs_delta(a_v, h_v)
        d_lo_n, d_hi_n = bootstrap_cliffs_delta_ci(a_v, h_v)
        knn_rows.append({
            'metric': col, 'human_mean': float(np.mean(h_v)), 'ai_mean': float(np.mean(a_v)),
            'mw_p': p_n, 'cliffs_delta': d_n,
            'cliffs_delta_ci_lo': d_lo_n, 'cliffs_delta_ci_hi': d_hi_n
        })

if knn_rows:
    knn_df_out = pd.DataFrame(knn_rows)
    _, knn_df_out['p_holm'], _, _ = multipletests(knn_df_out['mw_p'], method='holm')
    knn_df_out['mw_sig'] = knn_df_out['mw_p'].apply(fmt_p)
    knn_df_out.to_csv(TABLES_DIR_AI / 'novelty_knn_density_human_vs_allai.csv', index=False)
    print(knn_df_out[['metric', 'human_mean', 'ai_mean', 'mw_p', 'p_holm', 'cliffs_delta']].to_string())
else:
    print('WARNING: No novelty kNN/density data available')
    knn_df_out = None


In [ ]:
# Novelty figure: 3-panel boxplots if data available
_have_el   = novelty_element_df is not None and 'element_novel_0' in novelty_element_df.columns
_have_knn  = novelty_knn_df is not None and 'mean_knn_10' in novelty_knn_df.columns
_have_norm = novelty_norm_df is not None and 'novelty_z' in novelty_norm_df.columns

n_panels = sum([_have_el, _have_knn, _have_norm])
if n_panels == 0:
    print('WARNING: No novelty data for figure')
else:
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]
    fig.suptitle('Novelty Metrics\nHuman vs All-AI (per proposal)', fontsize=13)
    panel_i = 0

    if _have_el:
        col = 'element_novel_0'
        h_df = novelty_element_df[novelty_element_df['group_binary'] == 'Human'].copy()
        a_df = novelty_element_df[novelty_element_df['group_binary'] == 'AI'].copy()
        h_v = h_df[col].values
        a_v = a_df[col].values
        draw_boxplot_with_jitter(
            axes[panel_i], [h_v, a_v],
            [f'Human (n={len(h_v)})', f'All AI (n={len(a_v)})'],
            [HUMAN_COLOR, AI_COLOR],
            point_meta_list=[h_df, a_df],
            show_metadata_legend=True,
            metadata_legend_loc='best',
        )
        _, p_tmp = mannwhitneyu(a_v, h_v, alternative='two-sided')
        axes[panel_i].set_ylabel('Element novelty percentile (k=0)')
        axes[panel_i].set_title(f'Element Novelty (k=0)\n{fmt_p(p_tmp)}')
        panel_i += 1

    if _have_knn:
        col = 'mean_knn_10'
        h_df = novelty_knn_df[novelty_knn_df['group_binary'] == 'Human'].copy()
        a_df = novelty_knn_df[novelty_knn_df['group_binary'] == 'AI'].copy()
        h_v = h_df[col].values
        a_v = a_df[col].values
        draw_boxplot_with_jitter(
            axes[panel_i], [h_v, a_v],
            [f'Human (n={len(h_v)})', f'All AI (n={len(a_v)})'],
            [HUMAN_COLOR, AI_COLOR],
            point_meta_list=[h_df, a_df],
            show_metadata_legend=True,
            metadata_legend_loc='best',
        )
        _, p_tmp = mannwhitneyu(a_v, h_v, alternative='two-sided')
        axes[panel_i].set_ylabel('Mean kNN distance to literature (k=10)')
        axes[panel_i].set_title(f'Mean kNN Distance (k=10)\n{fmt_p(p_tmp)}')
        panel_i += 1

    if _have_norm:
        col = 'novelty_z'
        h_df = novelty_norm_df[novelty_norm_df['group_binary'] == 'Human'].copy()
        a_df = novelty_norm_df[novelty_norm_df['group_binary'] == 'AI'].copy()
        h_v = h_df[col].values
        a_v = a_df[col].values
        draw_boxplot_with_jitter(
            axes[panel_i], [h_v, a_v],
            [f'Human (n={len(h_v)})', f'All AI (n={len(a_v)})'],
            [HUMAN_COLOR, AI_COLOR],
            point_meta_list=[h_df, a_df],
            show_metadata_legend=True,
            metadata_legend_loc='best',
        )
        _, p_tmp = mannwhitneyu(a_v, h_v, alternative='two-sided')
        axes[panel_i].set_ylabel('Novelty z-score (local density normalized)')
        axes[panel_i].set_title(f'Novelty Z-score\n{fmt_p(p_tmp)}')
        panel_i += 1

    plt.tight_layout()
    fig.savefig(FIGURES_DIR_AI / 'novelty_human_vs_allai.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved novelty_human_vs_allai.png')


## Step 7B: Literature-Space Outliers

In [ ]:
outlier_cols = []
if master_df is not None:
    out_candidates = ['is_lit_outlier_mean10', 'is_lit_outlier_element0', 'is_lit_outlier_z']
    for col in out_candidates:
        if col in master_df.columns:
            outlier_cols.append(col)

if not outlier_cols:
    # Try loading individual CSVs
    _out_paths = [
        TABLES_DIR / 'literature_space_outliers_mean_knn_k10.csv',
        TABLES_DIR / 'literature_space_outliers_z.csv'
    ]
    for p in _out_paths:
        if p.exists():
            _odf = pd.read_csv(p)
            print(f'Loaded: {p.name}, columns: {_odf.columns.tolist()}')
    if not outlier_cols:
        print('WARNING: No literature-space outlier data found — skipping Step 7B')

if outlier_cols and master_df is not None:
    out_rows = []
    for col in outlier_cols:
        h_mask = master_df.index.isin(human_idx) if 'proposal_uid' not in master_df.columns else \
            master_df['group_binary'] == 'Human'
        a_mask = master_df.index.isin(ai_idx) if 'proposal_uid' not in master_df.columns else \
            master_df['group_binary'] == 'AI'
        h_out = master_df[h_mask][col].fillna(0).astype(int)
        a_out = master_df[a_mask][col].fillna(0).astype(int)
        ct = np.array([
            [h_out.sum(), (1 - h_out).sum()],
            [a_out.sum(), (1 - a_out).sum()]
        ])
        _, p_f = fisher_exact(ct)
        out_rows.append({
            'metric': col,
            'human_outlier_n': int(h_out.sum()),
            'ai_outlier_n': int(a_out.sum()),
            'human_outlier_frac': float(h_out.mean()),
            'ai_outlier_frac': float(a_out.mean()),
            'fisher_p': p_f
        })
    out_df = pd.DataFrame(out_rows)
    if len(out_df) > 0:
        _, out_df['p_holm'], _, _ = multipletests(out_df['fisher_p'], method='holm')
    out_df.to_csv(TABLES_DIR_AI / 'literature_space_outliers_human_vs_allai.csv', index=False)
    print(out_df.to_string())


## Analysis 3.5: Literature-Anchored UMAP with Embedding-Native Topic Regions


In [ ]:
print('='*85)
print('ANALYSIS 3.5: LITERATURE-ANCHORED UMAP - BERTOPIC EMBEDDING REGIONS')
print('='*85)
print('Reusing prepare_data_for_analysis literature UMAP/BERTopic artifacts; no literature UMAP or BERTopic refit is performed here.')

import textwrap
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

_lit_embed_dir_35 = PROJECT_ROOT / 'data' / 'embeddings' / 'literature'
_lit_umap2d_path_35 = _lit_embed_dir_35 / 'lit_umap2d.npy'
_lit_umap_reducer_path_35 = _lit_embed_dir_35 / 'lit_umap_reducer.pkl'
_prop_section1_cache_35 = _lit_embed_dir_35 / 'proposals_section1_lit_umap2d.npy'
_abs_emb_path_35 = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_rephrased_abstract.pkl'
_bt_assign_path_35 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_assignments.csv'
_bt_info_path_35 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_topic_info.csv'

_required_35 = [_lit_umap2d_path_35, _bt_assign_path_35, _bt_info_path_35]
_missing_35 = [p for p in _required_35 if not p.exists()]
_missing_projection_35 = (not _prop_section1_cache_35.exists()) and (lit_umap_reducer is None) and (not _lit_umap_reducer_path_35.exists())
if _missing_projection_35:
    _missing_35.append(_lit_umap_reducer_path_35)

if _missing_35:
    print(f'WARNING: Missing BERTopic literature-region outputs: {[str(p) for p in _missing_35]}')
    print('Run prepare_data_for_analysis.ipynb Sections 12 and 13 first. Skipping Analysis 3.5.')
elif D_pl_sorted_idx is None:
    print('WARNING: D_pl_sorted_idx is unavailable. Run the shared cache/loading cell with literature embeddings first.')
else:
    lit_2d_35 = lit_umap2d if lit_umap2d is not None else np.load(str(_lit_umap2d_path_35))

    # Project proposals using the same abstract-only embeddings used for literature novelty.
    # If proposal_embeddings_rephrased_abstract.pkl is newer than the projection cache,
    # recompute so corrected abstracts immediately update this visualization.
    cache_is_fresh_35 = (
        _prop_section1_cache_35.exists()
        and (not _abs_emb_path_35.exists() or _prop_section1_cache_35.stat().st_mtime >= _abs_emb_path_35.stat().st_mtime)
    )
    if cache_is_fresh_35:
        prop_section1_2d_35 = np.load(str(_prop_section1_cache_35))
        print(f'Loaded cached abstract-proposal projection: {_prop_section1_cache_35}')
    else:
        if 'X_prop_nov' in globals() and X_prop_nov.shape[0] == len(proposal_meta):
            X_prop_lit_35 = np.asarray(X_prop_nov, dtype=np.float32)
        elif _abs_emb_path_35.exists():
            with open(_abs_emb_path_35, 'rb') as f:
                _abs_raw_35 = pickle.load(f)
            if isinstance(_abs_raw_35, dict) and 'human_embeddings' in _abs_raw_35:
                _ha_35 = np.asarray(_abs_raw_35['human_embeddings'], dtype=np.float32)
                _aa_35 = np.asarray(_abs_raw_35['ai_embeddings'], dtype=np.float32)
                X_prop_lit_35 = np.vstack([_ha_35, _aa_35])
            elif isinstance(_abs_raw_35, dict) and 'embeddings' in _abs_raw_35:
                X_prop_lit_35 = np.asarray(_abs_raw_35['embeddings'], dtype=np.float32)
            else:
                X_prop_lit_35 = np.asarray(_abs_raw_35, dtype=np.float32)
            _pn_35 = np.linalg.norm(X_prop_lit_35, axis=1, keepdims=True)
            X_prop_lit_35 = (X_prop_lit_35 / np.clip(_pn_35, 1e-12, None)).astype(np.float32)
        else:
            X_prop_lit_35 = X_prop
            print('WARNING: abstract proposal embeddings not found; using full-proposal embeddings for projection.')

        if X_prop_lit_35.shape[0] != len(proposal_meta):
            raise RuntimeError(f'Proposal embedding rows ({X_prop_lit_35.shape[0]}) do not match proposal_meta ({len(proposal_meta)}).')
        if lit_umap_reducer is not None:
            reducer_35 = lit_umap_reducer
        else:
            with open(_lit_umap_reducer_path_35, 'rb') as f:
                reducer_35 = pickle.load(f)
        prop_section1_2d_35 = reducer_35.transform(X_prop_lit_35)
        np.save(str(_prop_section1_cache_35), prop_section1_2d_35)
        print(f'Saved refreshed abstract-proposal projection: {_prop_section1_cache_35}')

    proposals_section1_2d_umap = prop_section1_2d_35
    prop_lit_coords = prop_section1_2d_35
    if len(prop_section1_2d_35) != len(proposal_meta):
        raise RuntimeError(f'Proposal UMAP rows ({len(prop_section1_2d_35)}) do not match proposal_meta ({len(proposal_meta)}).')

    bt_df_35 = pd.read_csv(_bt_assign_path_35)
    bt_info_35 = pd.read_csv(_bt_info_path_35)
    if 'article_idx' in bt_df_35.columns:
        bt_df_35 = bt_df_35.set_index('article_idx').reindex(np.arange(len(lit_2d_35))).reset_index()
        if bt_df_35['bertopic_topic'].isna().any():
            missing_n_35 = int(bt_df_35['bertopic_topic'].isna().sum())
            raise RuntimeError(f'BERTopic assignments are missing {missing_n_35} article_idx rows needed to align with lit_umap2d.npy.')
    if len(bt_df_35) != len(lit_2d_35):
        raise RuntimeError(f'Literature UMAP rows ({len(lit_2d_35)}) do not match BERTopic assignments ({len(bt_df_35)}).')

    bt_topic_arr_35 = bt_df_35['bertopic_topic'].to_numpy(dtype=int)
    label_strategy_35 = str(bt_info_35.get('display_label_strategy', pd.Series(['unknown'])).dropna().iloc[0]) if len(bt_info_35) else 'unknown'
    print(f'BERTopic display-label strategy loaded from prepare_data: {label_strategy_35}')
    if label_strategy_35 != 'contrastive_phrase_v4':
        print('WARNING: BERTopic labels may be stale; rerun prepare_data Section 12 for latest contrastive labels.')

    bt_labels_35 = dict(zip(
        bt_info_35['Topic'].astype(int),
        bt_info_35.get('display_label', bt_info_35.get('Name', bt_info_35['Topic'].astype(str)))
    )) if len(bt_info_35) and 'Topic' in bt_info_35.columns else {}
    bt_counts_35 = bt_df_35['bertopic_topic'].value_counts().to_dict()
    region_ids_35 = sorted([int(t) for t in np.unique(bt_topic_arr_35) if int(t) >= 0])
    topic_rank_35 = sorted(region_ids_35, key=lambda t: bt_counts_35.get(t, 0), reverse=True)
    label_region_ids_35 = set(topic_rank_35[:25])
    cmap_35 = plt.get_cmap('tab20', max(len(region_ids_35), 1))
    region_color_35 = {t: cmap_35(i % cmap_35.N) for i, t in enumerate(region_ids_35)}

    top5_mask_35 = proposal_meta.get('is_top5_ranked', pd.Series(False, index=proposal_meta.index)).fillna(False).to_numpy(dtype=bool)
    funding_35 = pd.to_numeric(proposal_meta.get('funding', pd.Series(np.nan, index=proposal_meta.index)), errors='coerce').to_numpy()
    human_mask_35 = proposal_meta['group_binary'].eq('Human').to_numpy()
    ai_mask_35 = proposal_meta['group_binary'].eq('AI').to_numpy()
    marker_size_35 = np.where(top5_mask_35, 112, 78)

    def _style_literature_umap_axes_35(ax):
        x_pad = 0.03 * (float(np.nanmax(lit_2d_35[:, 0])) - float(np.nanmin(lit_2d_35[:, 0])))
        y_pad = 0.03 * (float(np.nanmax(lit_2d_35[:, 1])) - float(np.nanmin(lit_2d_35[:, 1])))
        ax.set_xlim(float(np.nanmin(lit_2d_35[:, 0])) - x_pad, float(np.nanmax(lit_2d_35[:, 0])) + x_pad)
        ax.set_ylim(float(np.nanmin(lit_2d_35[:, 1])) - y_pad, float(np.nanmax(lit_2d_35[:, 1])) + y_pad)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)

    def _draw_lit_background_35(ax, alpha=0.42, s=4, label_top_regions=True, label_fontsize=8, show_region_legend=False):
        outlier_mask = bt_topic_arr_35 == -1
        if outlier_mask.any():
            ax.scatter(lit_2d_35[outlier_mask, 0], lit_2d_35[outlier_mask, 1],
                       c='#d9d9d9', s=max(2, s - 1), alpha=0.18,
                       linewidths=0, rasterized=True, zorder=1)
        for t in region_ids_35:
            mask_t = bt_topic_arr_35 == t
            if mask_t.any():
                ax.scatter(lit_2d_35[mask_t, 0], lit_2d_35[mask_t, 1],
                           c=[region_color_35[t]], s=s, alpha=alpha,
                           linewidths=0, rasterized=True, zorder=2)
        if label_top_regions:
            for t in topic_rank_35:
                if t not in label_region_ids_35:
                    continue
                mask_t = bt_topic_arr_35 == t
                if mask_t.sum() < 20:
                    continue
                centroid_x, centroid_y = np.median(lit_2d_35[mask_t], axis=0)
                label = str(bt_labels_35.get(t, f'Region {t}'))
                label = ', '.join(label.split(',')[:4]).strip()
                label_text = textwrap.fill(f'{t}: {label}', width=34)
                txt = ax.text(
                    centroid_x, centroid_y, label_text,
                    fontsize=label_fontsize, fontweight='bold', ha='center', va='center', color='black',
                    bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white',
                          'edgecolor': region_color_35[t], 'linewidth': 1.2, 'alpha': 0.86},
                    zorder=6,
                )
                txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='white')])
        if show_region_legend:
            region_handles = [
                Line2D([0], [0], marker='o', color='w', markerfacecolor=region_color_35[t],
                       markersize=7, label=f'{t}: n={int(bt_counts_35.get(t, 0)):,}')
                for t in region_ids_35
            ]
            region_legend = ax.legend(handles=region_handles, title='BERTopic regions',
                                      loc='center left', bbox_to_anchor=(1.01, 0.5),
                                      frameon=False, fontsize=8, title_fontsize=9)
            ax.add_artist(region_legend)
        _style_literature_umap_axes_35(ax)

    def _scatter_group_35(ax, mask, color, label, marker='o', alpha=0.9, zorder=10):
        mask = np.asarray(mask, dtype=bool)
        if not mask.any():
            return []
        return scatter_proposal_points(ax, prop_section1_2d_35, mask, color, label,
                                       marker=marker, alpha=alpha, size=marker_size_35,
                                       zorder=zorder)

    def _draw_proposals_35(ax, group_filter=None, legend=True, legend_loc='upper right'):
        handles = []
        if group_filter in (None, 'Human'):
            handles += _scatter_group_35(ax, human_mask_35, HUMAN_COLOR,
                                         f'Human (n={int(human_mask_35.sum())})',
                                         marker='o', alpha=0.94, zorder=12)
        if group_filter in (None, 'All AI'):
            handles += _scatter_group_35(ax, ai_mask_35, AI_COLOR,
                                         f'All AI (n={int(ai_mask_35.sum())})',
                                         marker='D', alpha=0.84, zorder=10)
        if legend and handles:
            add_proposal_metadata_legend(ax, group_handles=handles, loc=legend_loc,
                                         title='Proposals', fontsize=8)

    fig35, ax35 = plt.subplots(1, 1, figsize=(14, 10))
    _draw_lit_background_35(ax35, alpha=0.42, s=4, label_top_regions=True, label_fontsize=8, show_region_legend=True)
    _draw_proposals_35(ax35, legend=True, legend_loc='upper right')
    ax35.set_title('Human vs All-AI Proposals in Fixed Literature Map - BERTopic Embedding Regions\nDots = literature colored by BioLinkBERT-derived region; markers = proposals', fontsize=13, fontweight='bold')
    ax35.set_xlabel('Literature UMAP Dim 1')
    ax35.set_ylabel('Literature UMAP Dim 2')
    plt.tight_layout()
    out_35 = FIGURES_DIR_AI / 'literature_umap_human_vs_allai.png'
    fig35.savefig(out_35, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_35}')

    def _split_proposal_sides_35():
        prop_x = prop_section1_2d_35[:, 0]
        if len(prop_x) < 2:
            split_x = float(np.median(prop_x)) if len(prop_x) else 0.0
        else:
            sorted_x = np.sort(prop_x)
            gaps = np.diff(sorted_x)
            split_x = float((sorted_x[int(np.argmax(gaps))] + sorted_x[int(np.argmax(gaps)) + 1]) / 2.0) if len(gaps) else float(np.median(prop_x))
        left_mask = prop_x <= split_x
        right_mask = prop_x > split_x
        if left_mask.sum() == 0 or right_mask.sum() == 0:
            split_x = float(np.median(prop_x))
            left_mask = prop_x <= split_x
            right_mask = prop_x > split_x
        return split_x, left_mask, right_mask

    def _zoom_limits_around_proposals_35(prop_mask, min_x_pad=0.95, min_y_pad=0.85):
        pts = prop_section1_2d_35[prop_mask]
        if len(pts) == 0:
            return None
        x_min, y_min = np.min(pts, axis=0)
        x_max, y_max = np.max(pts, axis=0)
        lit_x_span = float(np.ptp(lit_2d_35[:, 0])) or 1.0
        lit_y_span = float(np.ptp(lit_2d_35[:, 1])) or 1.0
        x_pad = max(min_x_pad, 0.55 * max(float(x_max - x_min), 1e-6), 0.035 * lit_x_span)
        y_pad = max(min_y_pad, 0.65 * max(float(y_max - y_min), 1e-6), 0.045 * lit_y_span)
        return (float(x_min - x_pad), float(x_max + x_pad)), (float(y_min - y_pad), float(y_max + y_pad))

    def _proposal_local_topic_ids_35(prop_mask, k_neighbors=35, min_topic_votes=1):
        pts = prop_section1_2d_35[prop_mask]
        if len(pts) == 0:
            return []
        k_neighbors = max(1, min(int(k_neighbors), len(lit_2d_35)))
        topic_votes = []
        for pt in pts:
            d2 = np.sum((lit_2d_35 - pt) ** 2, axis=1)
            nn = np.argpartition(d2, k_neighbors - 1)[:k_neighbors]
            nn_topics = bt_topic_arr_35[nn]
            nn_topics = nn_topics[nn_topics >= 0]
            if len(nn_topics) == 0:
                continue
            counts = pd.Series(nn_topics).value_counts()
            top_count = int(counts.iloc[0])
            for topic_id, count in counts.items():
                if int(count) >= max(min_topic_votes, int(0.25 * top_count)):
                    topic_votes.append(int(topic_id))
        if not topic_votes:
            return []
        return pd.Series(topic_votes).value_counts().index.astype(int).tolist()

    def _draw_lit_background_zoom_35(ax, xlim, ylim, prop_topic_ids, alpha=0.78, s=8, label_fontsize=7.2):
        visible_mask = (
            (lit_2d_35[:, 0] >= xlim[0]) & (lit_2d_35[:, 0] <= xlim[1]) &
            (lit_2d_35[:, 1] >= ylim[0]) & (lit_2d_35[:, 1] <= ylim[1])
        )
        prop_topic_ids = [int(t) for t in prop_topic_ids if int(t) >= 0]
        if not prop_topic_ids:
            local_counts = pd.Series(bt_topic_arr_35[visible_mask & (bt_topic_arr_35 >= 0)]).value_counts()
            prop_topic_ids = [int(t) for t in local_counts.head(4).index]
        for t in prop_topic_ids:
            mask_t = visible_mask & (bt_topic_arr_35 == int(t))
            if mask_t.any():
                ax.scatter(lit_2d_35[mask_t, 0], lit_2d_35[mask_t, 1],
                           c=[region_color_35.get(int(t), '#808080')], s=s, alpha=alpha,
                           linewidths=0, rasterized=True, zorder=2)
        for t in prop_topic_ids:
            mask_t = visible_mask & (bt_topic_arr_35 == int(t))
            if mask_t.sum() < 15:
                continue
            centroid_x, centroid_y = np.median(lit_2d_35[mask_t], axis=0)
            label = str(bt_labels_35.get(int(t), f'Region {int(t)}'))
            label = ', '.join(label.split(',')[:4]).strip()
            txt = ax.text(
                centroid_x, centroid_y, textwrap.fill(f'{int(t)}: {label}', width=28),
                fontsize=label_fontsize, fontweight='bold', ha='center', va='center', color='black',
                bbox={'boxstyle': 'round,pad=0.22', 'facecolor': 'white',
                      'edgecolor': region_color_35.get(int(t), '#808080'), 'linewidth': 1.1, 'alpha': 0.86},
                zorder=6, clip_on=True,
            )
            txt.set_path_effects([path_effects.withStroke(linewidth=2.2, foreground='white')])
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

    split_x_35, left_prop_mask_35, right_prop_mask_35 = _split_proposal_sides_35()
    zoom_specs_35 = [
        ('Left proposal-bearing literature regions', left_prop_mask_35),
        ('Right proposal-bearing literature regions', right_prop_mask_35),
    ]
    fig35_zoom, axes35_zoom = plt.subplots(1, 2, figsize=(17, 7.2), sharex=False, sharey=False)
    for ax_zoom, (pane_title_35, prop_mask_35) in zip(axes35_zoom, zoom_specs_35):
        zoom_lims_35 = _zoom_limits_around_proposals_35(prop_mask_35)
        prop_topic_ids_35 = _proposal_local_topic_ids_35(prop_mask_35)
        if zoom_lims_35 is not None:
            _draw_lit_background_zoom_35(ax_zoom, zoom_lims_35[0], zoom_lims_35[1], prop_topic_ids_35)
        _draw_proposals_35(ax_zoom, legend=False)
        if zoom_lims_35 is not None:
            ax_zoom.set_xlim(*zoom_lims_35[0])
            ax_zoom.set_ylim(*zoom_lims_35[1])
        topic_txt_35 = ', '.join([str(t) for t in prop_topic_ids_35]) if prop_topic_ids_35 else 'local'
        ax_zoom.set_title(f'{pane_title_35}\nproposal-region labels shown: {topic_txt_35}; n proposals={int(prop_mask_35.sum())}', fontsize=10.5, fontweight='bold')
        ax_zoom.set_xlabel('Literature UMAP Dim 1')
        ax_zoom.set_ylabel('Literature UMAP Dim 2')
        ax_zoom.grid(True, alpha=0.12, linestyle='--')
        for spine in ax_zoom.spines.values():
            spine.set_visible(True)
            spine.set_alpha(0.25)

    proposal_handles_35 = proposal_group_legend_handles(
        human_label=f'Human (n={int(human_mask_35.sum())})',
        ai_label=f'All AI (n={int(ai_mask_35.sum())})'
    ) + _metadata_legend_handles(include_funding=True, include_top5=True)
    fig35_zoom.legend(proposal_handles_35, [h.get_label() for h in proposal_handles_35], title='Proposals',
                      loc='center right', bbox_to_anchor=(1.01, 0.5),
                      fontsize=9, title_fontsize=10, framealpha=0.92)
    fig35_zoom.suptitle('Split Zoom: Proposal-Bearing BERTopic Regions in the Fixed Literature Map\nOnly local literature regions occupied by proposals are shown in each pane', fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.93, 0.91])
    out_35_zoom = FIGURES_DIR_AI / 'literature_umap_human_vs_allai_split.png'
    fig35_zoom.savefig(out_35_zoom, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved split zoom: {out_35_zoom}')
    print(f'Proposal x split for zoom panes: {split_x_35:.3f}; left n={int(left_prop_mask_35.sum())}, right n={int(right_prop_mask_35.sum())}')

    fig35b, axes35b = plt.subplots(1, 2, figsize=(17, 7.2), sharex=True, sharey=True)
    for ax, group_name in zip(axes35b, ['Human', 'All AI']):
        _draw_lit_background_35(ax, alpha=0.24, s=3, label_top_regions=False)
        _draw_proposals_35(ax, group_filter=group_name, legend=True, legend_loc='upper right')
        ax.set_title(group_name, fontsize=11, fontweight='bold')
        ax.set_xlabel('Literature UMAP Dim 1')
        ax.set_ylabel('Literature UMAP Dim 2')
    fig35b.suptitle('Proposal Locations by Group over BERTopic Literature Regions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_35b = FIGURES_DIR_AI / 'literature_umap_bertopic_by_group_human_vs_allai.png'
    fig35b.savefig(out_35b, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_35b}')

print('Analysis 3.5 complete.')


## Analysis 3.6: BERTopic Region Coverage

Recomputed from the current abstract-only proposal-to-literature neighbors, with AI models pooled as one All-AI group.


In [ ]:
print('='*85)
print('ANALYSIS 3.6: LITERATURE EMBEDDING-REGION COVERAGE - HUMAN VS ALL-AI')
print('='*85)

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D

_bt_assign_path_36 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_assignments.csv'
_bt_info_path_36 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_topic_info.csv'
K_NEIGHBORS_36 = 20
GROUP_ORDER_36 = ['Human', 'All AI']

if D_pl_sorted_idx is None:
    print('WARNING: D_pl_sorted_idx is unavailable. Run the shared cache/loading cell with literature embeddings first. Skipping Analysis 3.6.')
    prop_region_weights = None
elif not _bt_assign_path_36.exists():
    print('WARNING: lit_bertopic_assignments.csv not found. Run prepare_data Section 12. Skipping primary BERTopic region coverage.')
    prop_region_weights = None
else:
    bt_df_36 = pd.read_csv(_bt_assign_path_36)
    bt_info_36 = pd.read_csv(_bt_info_path_36) if _bt_info_path_36.exists() else pd.DataFrame()

    n_lit_36 = X_lit.shape[0] if 'X_lit' in globals() and X_lit is not None else D_pl_sorted_idx.shape[1]
    if 'article_idx' in bt_df_36.columns:
        bt_df_36 = bt_df_36.set_index('article_idx').reindex(np.arange(n_lit_36)).reset_index()
        if bt_df_36['bertopic_topic'].isna().any():
            missing_n_36 = int(bt_df_36['bertopic_topic'].isna().sum())
            raise RuntimeError(f'BERTopic assignments are missing {missing_n_36} article_idx rows needed to align with literature embeddings.')
    if len(bt_df_36) != n_lit_36:
        raise RuntimeError(f'BERTopic assignment rows ({len(bt_df_36)}) do not match literature embeddings ({n_lit_36}).')

    label_strategy_36 = str(bt_info_36.get('display_label_strategy', pd.Series(['unknown'])).dropna().iloc[0]) if len(bt_info_36) else 'unknown'
    print(f'BERTopic display-label strategy loaded from prepare_data: {label_strategy_36}')
    if label_strategy_36 != 'contrastive_phrase_v4':
        print('WARNING: BERTopic topic-info labels may be stale. Rerun prepare_data Section 12 for latest contrastive labels.')

    lit_region_arr_36 = bt_df_36['bertopic_topic'].to_numpy(dtype=int)
    region_ids_36 = sorted([int(t) for t in np.unique(lit_region_arr_36) if int(t) >= 0])
    region_to_col_36 = {t: i for i, t in enumerate(region_ids_36)}
    n_regions_36 = len(region_ids_36)
    if len(bt_info_36) and 'Topic' in bt_info_36.columns:
        region_label_36 = dict(zip(
            bt_info_36['Topic'].astype(int),
            bt_info_36.get('display_label', bt_info_36.get('Name', bt_info_36['Topic'].astype(str)))
        ))
    else:
        region_label_36 = {t: f'Region {t}' for t in region_ids_36}

    prop_region_weights = np.zeros((len(proposal_meta), n_regions_36), dtype=np.float32)
    prop_unassigned_frac_36 = np.zeros(len(proposal_meta), dtype=np.float32)
    for i in range(len(proposal_meta)):
        nbr_idx = D_pl_sorted_idx[i, :K_NEIGHBORS_36]
        nbr_regions = lit_region_arr_36[nbr_idx]
        valid = nbr_regions >= 0
        prop_unassigned_frac_36[i] = 1.0 - (valid.sum() / max(1, len(nbr_regions)))
        if valid.sum() > 0:
            for r in nbr_regions[valid]:
                prop_region_weights[i, region_to_col_36[int(r)]] += 1.0
            prop_region_weights[i] /= valid.sum()

    prop_region_max_weight_36 = prop_region_weights.max(axis=1) if n_regions_36 else np.zeros(len(proposal_meta))
    prop_region_entropy_36 = np.array([
        float(-np.sum(row[row > 0] * np.log(row[row > 0] + 1e-12))) if row.sum() > 0 else 0.0
        for row in prop_region_weights
    ])
    prop_effective_regions_36 = np.exp(prop_region_entropy_36)
    prop_region_n_covered_36 = (prop_region_weights > 0.05).sum(axis=1)
    prop_region_dominant_36 = np.array([
        region_ids_36[int(row.argmax())] if row.sum() > 0 and n_regions_36 else -1
        for row in prop_region_weights
    ], dtype=int)

    prop_region_cov_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
    prop_region_cov_df['group_all_ai'] = np.where(prop_region_cov_df['group_binary'].eq('Human'), 'Human', 'All AI')
    prop_region_cov_df['dominant_region'] = prop_region_dominant_36
    prop_region_cov_df['dominant_region_label'] = [region_label_36.get(int(r), 'mixed_or_unassigned' if int(r) == -1 else f'Region {int(r)}') for r in prop_region_dominant_36]
    prop_region_cov_df['max_region_weight'] = prop_region_max_weight_36
    prop_region_cov_df['region_entropy'] = prop_region_entropy_36
    prop_region_cov_df['effective_region_count'] = prop_effective_regions_36
    prop_region_cov_df['n_regions_gt5pct'] = prop_region_n_covered_36
    prop_region_cov_df['unassigned_neighbor_frac'] = prop_unassigned_frac_36
    for region_id, col in region_to_col_36.items():
        prop_region_cov_df[f'region_{region_id}_weight'] = prop_region_weights[:, col]
    prop_region_cov_df.to_csv(TABLES_DIR_AI / 'bertopic_region_coverage_per_proposal_human_vs_allai.csv', index=False)
    bertopic_proposal_df = prop_region_cov_df.copy()

    group_masks_36 = {
        'Human': proposal_meta['group_binary'].eq('Human').to_numpy(),
        'All AI': proposal_meta['group_binary'].eq('AI').to_numpy(),
    }
    group_distributions_36 = {}
    summary_rows_36 = []
    for group_name in GROUP_ORDER_36:
        gm_mask = group_masks_36[group_name]
        gm_w = prop_region_weights[gm_mask]
        gd = gm_w.sum(axis=0) if len(gm_w) else np.zeros(n_regions_36, dtype=float)
        if gd.sum() > 0:
            gd = gd / gd.sum()
        group_distributions_36[group_name] = gd
        entropy_val = float(-np.sum(gd[gd > 0] * np.log(gd[gd > 0] + 1e-12))) if gd.sum() > 0 else 0.0
        row36 = {
            'group': group_name,
            'n_proposals': int(gm_mask.sum()),
            'breadth_gt5pct': int((gd > 0.05).sum()),
            'shannon_entropy': round(entropy_val, 4),
            'effective_region_count': round(float(np.exp(entropy_val)), 4),
            'dominant_region_frac': round(float(gd.max()), 4) if len(gd) else np.nan,
            'mean_unassigned_neighbor_frac': round(float(prop_unassigned_frac_36[gm_mask].mean()), 4) if gm_mask.any() else np.nan,
        }
        for region_id, col in region_to_col_36.items():
            row36[f'region_{region_id}_weight'] = round(float(gd[col]), 4)
        summary_rows_36.append(row36)
    bertopic_region_cov_group_df = pd.DataFrame(summary_rows_36)

    # N-sensitive breadth diagnostic: compare Human n=23 with bootstrap All-AI samples of n=23.
    if 'boot_ai_idx_samples' in globals() and len(boot_ai_idx_samples):
        boot_breadth_vals_36 = np.array([
            float(pd.Series(prop_region_dominant_36[sample]).loc[lambda x: x >= 0].nunique())
            for sample in boot_ai_idx_samples
        ])
        ai_boot_breadth_mean_36 = float(np.mean(boot_breadth_vals_36))
        ai_boot_breadth_ci_36 = boot_ci(boot_breadth_vals_36)
    else:
        boot_breadth_vals_36 = np.array([])
        ai_boot_breadth_mean_36 = np.nan
        ai_boot_breadth_ci_36 = (np.nan, np.nan)
    human_breadth_36 = int(pd.Series(prop_region_dominant_36[group_masks_36['Human']]).loc[lambda x: x >= 0].nunique())
    ai_full_breadth_36 = int(pd.Series(prop_region_dominant_36[group_masks_36['All AI']]).loc[lambda x: x >= 0].nunique())
    bertopic_region_cov_group_df['dominant_region_breadth_human'] = human_breadth_36
    bertopic_region_cov_group_df['dominant_region_breadth_ai_full'] = ai_full_breadth_36
    bertopic_region_cov_group_df['dominant_region_breadth_ai_boot_mean'] = ai_boot_breadth_mean_36
    bertopic_region_cov_group_df['dominant_region_breadth_ai_boot_ci_lo'] = ai_boot_breadth_ci_36[0]
    bertopic_region_cov_group_df['dominant_region_breadth_ai_boot_ci_hi'] = ai_boot_breadth_ci_36[1]
    bertopic_region_cov_group_df.to_csv(TABLES_DIR_AI / 'bertopic_region_coverage_group_human_vs_allai.csv', index=False)

    h_mask_36 = group_masks_36['Human']
    a_mask_36 = group_masks_36['All AI']
    metric_specs_36 = [
        ('max_region_weight', prop_region_max_weight_36, 'Max region wt'),
        ('region_entropy', prop_region_entropy_36, 'Region entropy'),
        ('effective_region_count', prop_effective_regions_36, 'Effective regions'),
        ('n_regions_gt5pct', prop_region_n_covered_36, 'Regions >5%'),
        ('unassigned_neighbor_frac', prop_unassigned_frac_36, 'Unassigned frac'),
    ]
    test_rows_36 = []
    for metric_name, values, label in metric_specs_36:
        h_vals = np.asarray(values[h_mask_36], dtype=float)
        a_vals = np.asarray(values[a_mask_36], dtype=float)
        h_vals = h_vals[np.isfinite(h_vals)]
        a_vals = a_vals[np.isfinite(a_vals)]
        if len(h_vals) >= 2 and len(a_vals) >= 2:
            stat36, p36 = mannwhitneyu(a_vals, h_vals, alternative='two-sided')
            d36 = cliffs_delta(a_vals, h_vals)
            lo36, hi36 = bootstrap_cliffs_delta_ci(a_vals, h_vals)
        else:
            stat36, p36, d36, lo36, hi36 = np.nan, np.nan, np.nan, np.nan, np.nan
        test_rows_36.append({
            'comparison': 'All AI vs Human',
            'metric': metric_name,
            'label': label,
            'n_group': int(len(a_vals)),
            'n_human': int(len(h_vals)),
            'mean_group': float(np.mean(a_vals)) if len(a_vals) else np.nan,
            'mean_human': float(np.mean(h_vals)) if len(h_vals) else np.nan,
            'human_mean': float(np.mean(h_vals)) if len(h_vals) else np.nan,
            'ai_mean': float(np.mean(a_vals)) if len(a_vals) else np.nan,
            'u_stat': float(stat36) if np.isfinite(stat36) else np.nan,
            'mw_p': float(p36) if np.isfinite(p36) else np.nan,
            'p_value': float(p36) if np.isfinite(p36) else np.nan,
            'cliffs_delta': float(d36) if np.isfinite(d36) else np.nan,
            'cliffs_delta_ci_lo': float(lo36) if np.isfinite(lo36) else np.nan,
            'cliffs_delta_ci_hi': float(hi36) if np.isfinite(hi36) else np.nan,
            'breadth_human': human_breadth_36,
            'breadth_ai_full': ai_full_breadth_36,
            'breadth_ai_boot_mean': ai_boot_breadth_mean_36,
            'breadth_ai_boot_ci_lo': ai_boot_breadth_ci_36[0],
            'breadth_ai_boot_ci_hi': ai_boot_breadth_ci_36[1],
        })
    bertopic_region_tests_df = pd.DataFrame(test_rows_36)
    valid_p_36 = bertopic_region_tests_df['p_value'].notna()
    bertopic_region_tests_df['p_holm'] = np.nan
    if valid_p_36.any():
        _, p_holm36, _, _ = multipletests(bertopic_region_tests_df.loc[valid_p_36, 'p_value'], method='holm')
        bertopic_region_tests_df.loc[valid_p_36, 'p_holm'] = p_holm36
    bertopic_region_tests_df.to_csv(TABLES_DIR_AI / 'bertopic_region_coverage_human_vs_allai.csv', index=False)

    represented_regions_36 = [
        r for r in region_ids_36
        if any(group_distributions_36[g][region_to_col_36[r]] > 0 for g in GROUP_ORDER_36)
    ]
    represented_regions_36 = sorted(
        represented_regions_36,
        key=lambda r: sum(group_distributions_36[g][region_to_col_36[r]] for g in GROUP_ORDER_36),
        reverse=True,
    )
    cmap36 = plt.get_cmap('tab20', max(len(represented_regions_36), 1))
    region_colors_36 = {r: cmap36(i % cmap36.N) for i, r in enumerate(represented_regions_36)}

    fig36, (axL36, axR36) = plt.subplots(1, 2, figsize=(16, 6.6), gridspec_kw={'width_ratios': [1.25, 1.0]})
    x36 = np.arange(len(GROUP_ORDER_36))
    bottoms36 = np.zeros(len(GROUP_ORDER_36))
    for region_id in represented_regions_36:
        vals = np.array([group_distributions_36[g][region_to_col_36[region_id]] for g in GROUP_ORDER_36])
        if vals.sum() <= 0:
            continue
        label = str(region_label_36.get(region_id, f'Region {region_id}'))
        axL36.bar(x36, vals, bottom=bottoms36, color=region_colors_36[region_id],
                  edgecolor='white', linewidth=0.4, label=f'R{region_id}: {label}')
        bottoms36 += vals
    axL36.set_xticks(x36)
    axL36.set_xticklabels([f'Human\n(n={int(h_mask_36.sum())})', f'All AI\n(n={int(a_mask_36.sum())})'])
    axL36.set_ylabel('Proportion of BERTopic region weight')
    axL36.set_ylim(0, 1.0)
    axL36.set_title(f'Literature-region footprint\n(k={K_NEIGHBORS_36} nearest literature neighbors)', fontsize=11, fontweight='bold')
    axL36.grid(alpha=0.2, axis='y')

    legend_handles_36 = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor=region_colors_36[r],
               markersize=8, label=f'R{r}: {str(region_label_36.get(r, f"Region {r}"))[:46]}')
        for r in represented_regions_36
    ]
    if legend_handles_36:
        axL36.legend(handles=legend_handles_36, loc='upper center', bbox_to_anchor=(0.5, -0.14),
                     ncol=2, frameon=False, fontsize=8, title='Represented BERTopic regions', title_fontsize=9)

    axR36.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    plot_df36 = bertopic_region_tests_df.copy()
    y36 = np.arange(len(plot_df36))
    for yi, row in plot_df36.iterrows():
        d36 = row['cliffs_delta']
        lo36 = row['cliffs_delta_ci_lo']
        hi36 = row['cliffs_delta_ci_hi']
        if np.isfinite(d36) and np.isfinite(lo36) and np.isfinite(hi36):
            axR36.errorbar(d36, yi, xerr=np.array([[d36 - lo36], [hi36 - d36]]),
                           fmt='o', color=AI_COLOR, ecolor=AI_COLOR,
                           elinewidth=1.2, capsize=3, markersize=7)
            axR36.text(1.06, yi, f'MW(Holm)={fmt_p(row["p_holm"])}',
                       va='center', ha='left', fontsize=9, clip_on=False)
    axR36.set_yticks(y36)
    axR36.set_yticklabels(plot_df36['label'].values, fontsize=9)
    axR36.invert_yaxis()
    axR36.set_xlim(-1.05, 1.05)
    axR36.set_xlabel("Cliff's delta (bootstrap 95% CI)\npositive = All AI > Human", fontsize=10)
    axR36.set_title('Effect size + MW significance', fontsize=11, fontweight='bold')
    axR36.grid(alpha=0.2, axis='x')

    fig36.suptitle('BERTopic Literature-Region Coverage - Human vs All-AI', fontsize=13, fontweight='bold')
    fig36.subplots_adjust(bottom=0.30, wspace=0.42)
    out_36 = FIGURES_DIR_AI / 'bertopic_region_coverage_human_vs_allai.png'
    fig36.savefig(out_36, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_36}')
    print('\nGroup-level BERTopic region coverage:')
    print(bertopic_region_cov_group_df[['group', 'breadth_gt5pct', 'shannon_entropy', 'effective_region_count', 'dominant_region_frac', 'mean_unassigned_neighbor_frac']].to_string(index=False))
    print(f'\nDominant-region breadth: Human={human_breadth_36}, All AI full={ai_full_breadth_36}, All AI boot n={N_SUBSAMPLE} mean={ai_boot_breadth_mean_36:.2f} [{ai_boot_breadth_ci_36[0]:.2f}, {ai_boot_breadth_ci_36[1]:.2f}]')
    print('\nMW tests vs Human:')
    print(bertopic_region_tests_df[['comparison', 'metric', 'mean_group', 'mean_human', 'p_value', 'p_holm']].to_string(index=False))

print('Analysis 3.6 complete.')


## Analysis 3.7: MeSH Term Coverage

In [ ]:
# Build per-proposal MeSH sets from nearest literature neighbors
if D_pl_sorted_idx is not None and articles:
    k_mesh = 20
    mesh_per_prop = []
    for i in range(len(D_pp)):
        nn_arts = D_pl_sorted_idx[i, :k_mesh]
        mesh_set = set()
        for art_idx in nn_arts:
            if art_idx < len(articles):
                mt = articles[art_idx].get('mesh_terms', [])
                if mt:
                    mesh_set.update(mt)
        mesh_per_prop.append(mesh_set)
    mesh_counts = np.array([len(s) for s in mesh_per_prop])
    print(f'Built MeSH sets from {k_mesh} nearest lit neighbors. Mean count: {np.mean(mesh_counts):.1f}')
elif mesh_proposal_df is not None:
    # Fallback: use precomputed per-proposal counts
    mesh_counts = np.zeros(len(D_pp))
    for _, row_m in mesh_proposal_df.iterrows():
        # match by proposal_uid or title
        if 'proposal_uid' in mesh_proposal_df.columns:
            match = proposal_meta[proposal_meta['proposal_uid'] == row_m['proposal_uid']]
        else:
            match = proposal_meta[proposal_meta['title'] == row_m['title']]
        if len(match) > 0:
            mesh_counts[match.index[0]] = row_m.get('unique_mesh_count', 0)
    mesh_per_prop = None
    print('Using precomputed MeSH counts from mesh_proposal_df')
else:
    mesh_per_prop = None
    mesh_counts = None
    print('WARNING: No MeSH data available — skipping Analysis 3.7')

if mesh_counts is not None:
    human_mesh_counts = mesh_counts[human_idx]
    ai_full_mesh_counts = mesh_counts[ai_idx]

    stat_mesh, p_mw_mesh = mannwhitneyu(ai_full_mesh_counts, human_mesh_counts,
                                         alternative='two-sided')
    delta_mesh = cliffs_delta(ai_full_mesh_counts, human_mesh_counts)
    d_lo_mesh, d_hi_mesh = bootstrap_cliffs_delta_ci(ai_full_mesh_counts, human_mesh_counts)

    # Group-union MeSH (N-sensitive)
    if mesh_per_prop is not None:
        human_union_size = len(set().union(*[mesh_per_prop[i] for i in human_idx]))

        def _union_fn(s_idx):
            sets = [mesh_per_prop[i] for i in s_idx if i < len(mesh_per_prop)]
            return float(len(set().union(*sets))) if sets else 0.0

        boot_union_vals = np.array([_union_fn(s) for s in boot_ai_idx_samples])
        ai_full_union = len(set().union(*[mesh_per_prop[i] for i in ai_idx]))
        ai_boot_union_mean = float(np.mean(boot_union_vals))
        ai_boot_union_ci   = boot_ci(boot_union_vals)
    else:
        human_union_size = ai_full_union = ai_boot_union_mean = np.nan
        ai_boot_union_ci = (np.nan, np.nan)

    print(f'Human per-proposal MeSH: mean={np.mean(human_mesh_counts):.1f}')
    print(f'AI per-proposal MeSH:    mean={np.mean(ai_full_mesh_counts):.1f}')
    print(f"MW p={p_mw_mesh:.4f} ({fmt_p(p_mw_mesh)}), Cliff's d={delta_mesh:.3f}")
    if mesh_per_prop is not None:
        print(f'Human union MeSH: {human_union_size}')
        print(f'AI Boot union: {ai_boot_union_mean:.1f} [{ai_boot_union_ci[0]:.1f}, {ai_boot_union_ci[1]:.1f}]')

    pd.DataFrame([{
        'human_mesh_mean': float(np.mean(human_mesh_counts)),
        'ai_mesh_mean': float(np.mean(ai_full_mesh_counts)),
        'mw_p': p_mw_mesh, 'cliffs_delta': delta_mesh,
        'cliffs_delta_ci_lo': d_lo_mesh, 'cliffs_delta_ci_hi': d_hi_mesh,
        'human_union_size': human_union_size,
        'ai_full_union_size': ai_full_union,
        'ai_boot_union_mean': ai_boot_union_mean,
        'ai_boot_union_ci_lo': ai_boot_union_ci[0],
        'ai_boot_union_ci_hi': ai_boot_union_ci[1],
    }]).to_csv(TABLES_DIR_AI / 'mesh_coverage_human_vs_allai.csv', index=False)

    # Figure: 2-panel
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    fig.suptitle('MeSH Term Coverage\nHuman vs All-AI literature topic footprint', fontsize=12)

    draw_boxplot_with_jitter(
        axes[0],
        [human_mesh_counts, ai_full_mesh_counts],
        [f'Human (n={len(human_idx)})', f'All AI (n={len(ai_idx)})'],
        [HUMAN_COLOR, AI_COLOR],
        point_meta_list=[proposal_meta.loc[human_idx], proposal_meta.loc[ai_idx]],
        show_metadata_legend=True,
        metadata_legend_loc='best',
    )
    axes[0].set_ylabel('Unique MeSH terms (k=20 NN)')
    axes[0].set_title(f'Per-proposal MeSH count\nMW {fmt_p(p_mw_mesh)}')

    if mesh_per_prop is not None:
        axes[1].bar([0, 1],
                    [human_union_size, ai_boot_union_mean],
                    color=[HUMAN_COLOR, AI_COLOR], alpha=0.75, width=0.5)
        axes[1].errorbar(1, ai_boot_union_mean,
                         yerr=[[ai_boot_union_mean - ai_boot_union_ci[0]],
                               [ai_boot_union_ci[1] - ai_boot_union_mean]],
                         fmt='none', color='black', capsize=5, linewidth=1.5)
        axes[1].set_xticks([0, 1])
        axes[1].set_xticklabels(['Human', 'All AI\n[bootstrap n=23]'])
        axes[1].set_ylabel('Group-union MeSH term count')
        axes[1].set_title('Group-union MeSH breadth\n(bootstrap-corrected)')
        axes[1].yaxis.grid(True, alpha=0.25)
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR_AI / 'mesh_coverage_human_vs_allai.png',
                dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved mesh_coverage_human_vs_allai.png')


## Analysis 3.8: Publication Year Recency

In [ ]:

print('='*85)
print('ANALYSIS 3.8: PUBLICATION YEAR RECENCY OF NEAREST LITERATURE - HUMAN VS ALL-AI')
print('='*85)

import re as _re38
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

K_YEAR_38 = 20
GROUP_ORDER_38 = ['Human', 'All AI']
_bt_assign_path_38 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_assignments.csv'
_bt_info_path_38 = PROJECT_ROOT / 'data' / 'prepared' / condition / 'lit_bertopic_topic_info.csv'

def _parse_year_38(date_str):
    m = _re38.search(r'\b(\d{4})\b', str(date_str))
    return int(m.group(1)) if m else 0

if D_pl_sorted_idx is None or not articles:
    print('WARNING: literature neighbor indices/articles unavailable - skipping Analysis 3.8')
else:
    lit_years_arr_38 = np.array([_parse_year_38(art.get('publication_date', '')) for art in articles])

    prop_median_year_38 = []
    prop_mean_year_38 = []
    for i in range(len(proposal_meta)):
        nbr_idx = D_pl_sorted_idx[i, :K_YEAR_38]
        years = lit_years_arr_38[nbr_idx]
        valid_years = years[years > 0]
        prop_median_year_38.append(float(np.median(valid_years)) if len(valid_years) > 0 else np.nan)
        prop_mean_year_38.append(float(np.mean(valid_years)) if len(valid_years) > 0 else np.nan)
    prop_median_year_38 = np.array(prop_median_year_38)
    prop_mean_year_38 = np.array(prop_mean_year_38)

    region_label_map_38 = {}
    if _bt_info_path_38.exists():
        _bt_info_38 = pd.read_csv(_bt_info_path_38)
        if 'Topic' in _bt_info_38.columns:
            region_label_map_38 = dict(zip(
                _bt_info_38['Topic'].astype(int),
                _bt_info_38.get('display_label', _bt_info_38['Topic'].astype(str)),
            ))

    if 'prop_region_weights' in globals() and prop_region_weights is not None and prop_region_weights.shape[0] == len(proposal_meta):
        if 'prop_region_cov_df' in globals() and prop_region_cov_df is not None:
            _region_cols_38 = [int(c.replace('region_', '').replace('_weight', '')) for c in prop_region_cov_df.columns if c.startswith('region_') and c.endswith('_weight')]
        else:
            _region_cols_38 = []
        if len(_region_cols_38) != prop_region_weights.shape[1]:
            _region_cols_38 = list(range(prop_region_weights.shape[1]))
        _dom_col_38 = prop_region_weights.argmax(axis=1)
        _dom_w_38 = prop_region_weights.max(axis=1)
        prop_lit_region_stratum_38 = np.array([
            _region_cols_38[j] if _dom_w_38[i] >= 0.20 else -1
            for i, j in enumerate(_dom_col_38)
        ], dtype=int)
    elif _bt_assign_path_38.exists():
        _bt_df_38 = pd.read_csv(_bt_assign_path_38)
        n_lit_38 = D_pl_sorted_idx.shape[1]
        if 'article_idx' in _bt_df_38.columns:
            _bt_df_38 = _bt_df_38.set_index('article_idx').reindex(np.arange(n_lit_38)).reset_index()
        _region_arr_38 = _bt_df_38['bertopic_topic'].to_numpy(dtype=int)
        _region_ids_38 = sorted([int(t) for t in np.unique(_region_arr_38) if int(t) >= 0])
        _region_to_col_38 = {t: i for i, t in enumerate(_region_ids_38)}
        _weights_38 = np.zeros((len(proposal_meta), len(_region_ids_38)), dtype=np.float32)
        for i in range(len(proposal_meta)):
            nbr_idx = D_pl_sorted_idx[i, :K_YEAR_38]
            nbr_regions = _region_arr_38[nbr_idx]
            valid = nbr_regions >= 0
            if valid.sum() > 0:
                for r in nbr_regions[valid]:
                    _weights_38[i, _region_to_col_38[int(r)]] += 1.0
                _weights_38[i] /= valid.sum()
        _dom_col_38 = _weights_38.argmax(axis=1) if len(_region_ids_38) else np.zeros(len(proposal_meta), dtype=int)
        _dom_w_38 = _weights_38.max(axis=1) if len(_region_ids_38) else np.zeros(len(proposal_meta), dtype=float)
        prop_lit_region_stratum_38 = np.array([
            _region_ids_38[j] if len(_region_ids_38) and _dom_w_38[i] >= 0.20 else -1
            for i, j in enumerate(_dom_col_38)
        ], dtype=int)
    else:
        prop_lit_region_stratum_38 = np.full(len(proposal_meta), -1, dtype=int)
        print('WARNING: BERTopic region assignments not found; all proposals assigned to mixed_or_unassigned stratum.')

    prop_38_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
    prop_38_df['group_all_ai'] = np.where(prop_38_df['group_binary'].eq('Human'), 'Human', 'All AI')
    prop_38_df['median_neighbor_year'] = prop_median_year_38
    prop_38_df['mean_neighbor_year'] = np.round(prop_mean_year_38, 2)
    prop_38_df['lit_region_stratum'] = prop_lit_region_stratum_38
    prop_38_df['lit_region_label'] = [
        region_label_map_38.get(int(r), 'mixed_or_unassigned' if int(r) == -1 else f'Region {int(r)}')
        for r in prop_lit_region_stratum_38
    ]
    prop_38_df.to_csv(TABLES_DIR_AI / 'lit_neighbor_year_per_proposal_human_vs_allai.csv', index=False)

    strata_38 = sorted([int(s) for s in np.unique(prop_lit_region_stratum_38) if int(s) >= 0])
    test_rows_38 = []
    for stratum in strata_38:
        s_mask = prop_lit_region_stratum_38 == stratum
        h_vals = prop_median_year_38[s_mask & proposal_meta['group_binary'].eq('Human').to_numpy()]
        a_vals = prop_median_year_38[s_mask & proposal_meta['group_binary'].eq('AI').to_numpy()]
        h_vals = h_vals[~np.isnan(h_vals)]
        a_vals = a_vals[~np.isnan(a_vals)]
        if len(a_vals) >= 3 and len(h_vals) >= 3:
            stat38, p38 = mannwhitneyu(a_vals, h_vals, alternative='two-sided')
            test_rows_38.append({
                'stratum': stratum,
                'stratum_label': region_label_map_38.get(int(stratum), f'Region {stratum}'),
                'comparison': 'All AI vs Human',
                'n_group': len(a_vals),
                'n_human': len(h_vals),
                'median_year_group': float(np.median(a_vals)),
                'median_year_human': float(np.median(h_vals)),
                'mean_year_group': float(np.mean(a_vals)),
                'mean_year_human': float(np.mean(h_vals)),
                'u_stat': float(stat38),
                'p_value': float(p38),
                'cliffs_delta': cliffs_delta(a_vals, h_vals),
            })

    test_df_38 = pd.DataFrame(test_rows_38) if test_rows_38 else pd.DataFrame()
    if len(test_df_38) > 0:
        _, p_holm38, _, _ = multipletests(test_df_38['p_value'], method='holm')
        test_df_38['p_holm'] = p_holm38
    test_df_38.to_csv(TABLES_DIR_AI / 'lit_neighbor_year_human_vs_allai.csv', index=False)

    summary_rows_38 = []
    all_strata_38 = strata_38 + ([-1] if (-1 in prop_lit_region_stratum_38) else [])
    for group_name, gm_mask in [
        ('Human', proposal_meta['group_binary'].eq('Human').to_numpy()),
        ('All AI', proposal_meta['group_binary'].eq('AI').to_numpy()),
    ]:
        for stratum in all_strata_38:
            s_mask = prop_lit_region_stratum_38 == stratum
            vals = prop_median_year_38[gm_mask & s_mask]
            vals = vals[~np.isnan(vals)]
            if len(vals) > 0:
                summary_rows_38.append({
                    'group': group_name,
                    'stratum': stratum if stratum >= 0 else 'mixed_or_unassigned',
                    'stratum_label': region_label_map_38.get(int(stratum), 'mixed_or_unassigned' if stratum == -1 else f'Region {stratum}'),
                    'n': len(vals),
                    'mean_year': round(float(np.mean(vals)), 1),
                    'median_year': float(np.median(vals)),
                })
    pd.DataFrame(summary_rows_38).to_csv(TABLES_DIR_AI / 'lit_neighbor_year_region_group_summary_human_vs_allai.csv', index=False)

    _stratum_counts_38 = pd.Series(prop_lit_region_stratum_38).value_counts()
    plot_strata_38 = [int(s) for s in _stratum_counts_38.index if int(s) >= 0][:8]
    if -1 in _stratum_counts_38.index:
        plot_strata_38.append(-1)
    if not plot_strata_38:
        plot_strata_38 = [-1]

    n_cols_38 = min(len(plot_strata_38), 4)
    n_rows_38 = max(1, -(-len(plot_strata_38) // n_cols_38))
    fig38, axes38 = plt.subplots(n_rows_38, n_cols_38, figsize=(5 * n_cols_38, 5 * n_rows_38), squeeze=False)
    for ax in axes38.flat:
        ax.set_visible(False)

    for panel_i, stratum in enumerate(plot_strata_38):
        ax38 = axes38.flat[panel_i]
        ax38.set_visible(True)
        s_mask = prop_lit_region_stratum_38 == stratum
        label_txt = region_label_map_38.get(int(stratum), 'Mixed/Unassigned' if stratum == -1 else f'Region {stratum}')
        h_df38 = prop_38_df.loc[s_mask & prop_38_df['group_all_ai'].eq('Human')].copy()
        a_df38 = prop_38_df.loc[s_mask & prop_38_df['group_all_ai'].eq('All AI')].copy()
        box_vals_38 = [
            h_df38['median_neighbor_year'].dropna().to_numpy(dtype=float),
            a_df38['median_neighbor_year'].dropna().to_numpy(dtype=float),
        ]
        if any(len(v) > 0 for v in box_vals_38):
            draw_boxplot_with_jitter(
                ax38,
                box_vals_38,
                [f'Human\n(n={len(box_vals_38[0])})', f'All AI\n(n={len(box_vals_38[1])})'],
                [HUMAN_COLOR, AI_COLOR],
                point_meta_list=[h_df38, a_df38],
                show_metadata_legend=(panel_i == 0),
                metadata_legend_loc='best',
            )
        ax38.set_title((f'Region {stratum}: ' if stratum >= 0 else '') + str(label_txt)[:60], fontsize=9, fontweight='bold')
        ax38.set_ylabel(f'Median pub year (k={K_YEAR_38})', fontsize=8)
        ax38.grid(True, alpha=0.3, axis='y', linestyle='--')

    fig38.suptitle('Publication Year Recency of Nearest Literature - Within BERTopic Embedding Regions\nHuman vs All-AI; diamond/error bar = mean bootstrap 95% CI', fontsize=12, fontweight='bold')
    plt.tight_layout()
    out_38 = FIGURES_DIR_AI / 'lit_neighbor_year_by_group_within_bertopic_region_human_vs_allai.png'
    fig38.savefig(out_38, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_38}')
    if len(test_df_38) > 0:
        print('\nWithin-region MW tests:')
        print(test_df_38[['stratum', 'comparison', 'median_year_group', 'median_year_human', 'p_value', 'p_holm']].to_string(index=False))
    else:
        print('No within-region tests (insufficient group sizes per stratum).')
print('Analysis 3.8 complete.')


# PART IV: STYLE

In [ ]:
if style_df is None:
    print('WARNING: style_df not loaded — skipping Part IV')
else:
    style_features_list = [
        'n_words', 'n_chars', 'n_sents', 'avg_word_len', 'avg_sent_len_words',
        'type_token_ratio', 'stopword_rate', 'hedge_rate',
        'flesch_reading_ease', 'fk_grade_level',
        'comma_per_1k_chars', 'semicolon_per_1k_chars', 'colon_per_1k_chars',
        'dash_per_1k_chars', 'paren_per_1k_chars', 'quote_per_1k_chars',
        'newline_per_1k_chars', 'bullet_per_1k_chars'
    ]
    style_features_avail = [c for c in style_features_list if c in style_df.columns]

    # Determine group column
    style_group_col = 'group' if 'group' in style_df.columns else 'group_binary'
    if style_group_col not in style_df.columns:
        # Try merging with proposal_meta
        style_df = style_df.merge(proposal_meta[['title', 'group_binary']], on='title', how='left')
        style_group_col = 'group_binary'

    style_human = style_df[style_df[style_group_col] == 'Human']
    style_ai    = style_df[(style_df[style_group_col] == 'AI') |
                            (style_df[style_group_col].isin(['Claude', 'Gemini', 'GPT-5.2']))]
    if len(style_ai) == 0:
        # fallback
        is_ai_col = 'is_ai' if 'is_ai' in style_df.columns else None
        if is_ai_col:
            style_ai = style_df[style_df[is_ai_col] == 1]

    style_rows = []
    for col in style_features_avail:
        h_v = style_human[col].dropna().values
        a_v = style_ai[col].dropna().values
        if len(h_v) < 3 or len(a_v) < 3:
            continue
        stat_s, p_s = mannwhitneyu(a_v, h_v, alternative='two-sided')
        d_s = cliffs_delta(a_v, h_v)
        d_lo_s, d_hi_s = bootstrap_cliffs_delta_ci(a_v, h_v)
        style_rows.append({
            'feature': col,
            'human_mean': float(np.mean(h_v)), 'ai_mean': float(np.mean(a_v)),
            'mw_p': p_s, 'cliffs_delta': d_s,
            'cliffs_delta_ci_lo': d_lo_s, 'cliffs_delta_ci_hi': d_hi_s
        })
    style_out_df = pd.DataFrame(style_rows)
    if len(style_out_df) > 0:
        _, style_out_df['p_holm'], _, _ = multipletests(style_out_df['mw_p'], method='holm')
    style_out_df['mw_sig'] = style_out_df['mw_p'].apply(fmt_p)
    style_out_df.to_csv(TABLES_DIR_AI / 'style_features_human_vs_allai.csv', index=False)
    print(f'{(style_out_df["p_holm"] < 0.05).sum()} of {len(style_out_df)} '
          f'style features significant after Holm correction')
    print(style_out_df[['feature', 'human_mean', 'ai_mean', 'mw_p', 'p_holm', 'cliffs_delta']].to_string())

    # Style classifier (Human vs All-AI)
    if len(style_features_avail) > 0:
        from sklearn.preprocessing import StandardScaler
        style_all = pd.concat([style_human, style_ai], ignore_index=True)
        y_cls = np.array([1 if r[style_group_col] == 'Human' else 0
                          for _, r in style_all.iterrows()])
        X_cls = style_all[style_features_avail].fillna(0).values
        scaler = StandardScaler()
        X_cls_s = scaler.fit_transform(X_cls)

        clf = LogisticRegression(class_weight='balanced', max_iter=1000)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        auroc_scores = cross_val_score(clf, X_cls_s, y_cls, cv=cv, scoring='roc_auc')
        mean_auroc = float(np.mean(auroc_scores))

        # Permutation test for AUROC
        rng_cls = np.random.default_rng(42)
        perm_aurocs = []
        for _ in range(500):
            y_perm = rng_cls.permutation(y_cls)
            perm_aurocs.append(float(np.mean(
                cross_val_score(clf, X_cls_s, y_perm, cv=cv, scoring='roc_auc'))))
        p_perm_cls = (np.sum(np.array(perm_aurocs) >= mean_auroc) + 1) / 501

        print(f'\nStyle classifier AUROC: {mean_auroc:.3f} (5-fold CV), perm p={p_perm_cls:.4f}')
        pd.DataFrame([{
            'cv_auroc_mean': mean_auroc, 'cv_auroc_std': float(np.std(auroc_scores)),
            'perm_p': p_perm_cls, 'n_features': len(style_features_avail)
        }]).to_csv(TABLES_DIR_AI / 'style_only_classifier_human_vs_allai.csv', index=False)

        # Figure: forest plot + AUROC box
        fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(style_features_avail) * 0.5 + 2)))
        fig.suptitle('Style Features Analysis\nHuman vs All-AI writing style metrics', fontsize=12)

        # Forest plot
        ax_f2 = axes[0]
        ax_f2.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
        ys2 = list(range(len(style_out_df)))
        for yi2, row_s in style_out_df.iterrows():
            col_s = 'tomato' if row_s.get('p_holm', 1) < 0.05 else AI_COLOR
            ax_f2.errorbar(row_s['cliffs_delta'], yi2,
                           xerr=[[row_s['cliffs_delta'] - row_s['cliffs_delta_ci_lo']],
                                 [row_s['cliffs_delta_ci_hi'] - row_s['cliffs_delta']]],
                           fmt='o', color=col_s, markersize=6,
                           ecolor='black', elinewidth=1.0, capsize=3)
        ax_f2.set_yticks(ys2)
        ax_f2.set_yticklabels(style_out_df['feature'].values, fontsize=8)
        ax_f2.set_xlabel("Cliff's δ (bootstrap 95% CI)\npositive = AI > Human")
        ax_f2.set_title('Style feature effect sizes\n(red = Holm p<0.05)')
        ax_f2.yaxis.grid(True, alpha=0.25)

        # AUROC
        ax_r = axes[1]
        draw_boxplot_with_jitter(
            ax_r,
            [auroc_scores],
            ['Human vs All AI'],
            [AI_COLOR],
            show_metadata_legend=False,
        )
        ax_r.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
        ax_r.set_xticks([0])
        ax_r.set_xticklabels(['Human vs All AI'])
        ax_r.set_ylabel('AUROC (5-fold CV)')
        ax_r.set_title(f'Style-only classifier\nAUROC={mean_auroc:.3f}, perm {fmt_p(p_perm_cls)}')
        ax_r.legend()
        ax_r.yaxis.grid(True, alpha=0.25)

        plt.tight_layout()
        fig.savefig(FIGURES_DIR_AI / 'style_human_vs_allai.png',
                    dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print('Saved style_human_vs_allai.png')


## Finalize AI Heterogeneity Supplementary Table

In [ ]:
_hetero_out = TABLES_DIR_AI / 'supplementary_ai_heterogeneity_kruskal_wallis.csv'
if _hetero_out.exists():
    hetero_final = pd.read_csv(_hetero_out)
    valid_h = hetero_final['p_kw'].notna()
    if valid_h.sum() > 0:
        rej_f, p_holm_f, _, _ = multipletests(
            hetero_final.loc[valid_h, 'p_kw'].values, alpha=0.05, method='holm')
        hetero_final.loc[valid_h, 'p_holm_final'] = p_holm_f
        hetero_final.loc[valid_h, 'sig_holm_final'] = rej_f
    hetero_final.to_csv(_hetero_out, index=False)
    n_sig_f = int(hetero_final.get('sig_holm_final', pd.Series(False)).sum())
    print(f'Heterogeneity summary: {n_sig_f} of {len(hetero_final)} outcomes '
          f'significantly heterogeneous across AI models (Holm correction).')
    print(hetero_final.to_string())
else:
    print('WARNING: heterogeneity table not found')


## Unified Summary Export

In [ ]:
# Compile all primary results
summary_rows = []

# Diversity
summary_rows += [
    {'domain': 'Diversity', 'metric': 'Mean Pairwise Distance',
     'human_val': human_mpd, 'ai_boot_mean': ai_boot_mpd,
     'perm_p': p_perm_pp, 'cliffs_delta': delta, 'sig': fmt_p(p_perm_pp)},
    {'domain': 'Diversity', 'metric': 'LOO Centroid Distance',
     'human_val': human_loo_mean, 'ai_boot_mean': ai_boot_loo_mean,
     'perm_p': p_perm_loo, 'cliffs_delta': delta_loo, 'sig': fmt_p(p_perm_loo)},
    {'domain': 'Diversity', 'metric': 'Global Centroid Distance',
     'human_val': float(np.mean(human_gc)), 'ai_boot_mean': ai_boot_gc_mean,
     'perm_p': p_perm_gc, 'cliffs_delta': delta_gc, 'sig': fmt_p(p_perm_gc)},
    {'domain': 'Diversity', 'metric': 'MST Mean Edge Weight',
     'human_val': human_mst, 'ai_boot_mean': ai_boot_mst_mean,
     'perm_p': p_perm_mst, 'cliffs_delta': np.nan, 'sig': fmt_p(p_perm_mst)},
    {'domain': 'Diversity', 'metric': 'Sparseness (Medoid)',
     'human_val': human_spar, 'ai_boot_mean': ai_boot_spar_mean,
     'perm_p': p_perm_spar, 'cliffs_delta': delta_spar, 'sig': fmt_p(p_perm_spar)},
    {'domain': 'Diversity', 'metric': 'Chamfer (NN Distance)',
     'human_val': human_chamfer, 'ai_boot_mean': ai_boot_cham_mean,
     'perm_p': p_perm_nn, 'cliffs_delta': delta_nn, 'sig': fmt_p(p_perm_nn)},
    {'domain': 'Diversity', 'metric': 'Grid Entropy',
     'human_val': human_ent, 'ai_boot_mean': ai_boot_ent_mean,
     'perm_p': p_perm_ent, 'cliffs_delta': np.nan, 'sig': fmt_p(p_perm_ent)},
]

# Novelty (per-proposal MW p-values)
for _csv_name, _domain in [
    ('novelty_element_percentiles_human_vs_allai.csv', 'Novelty-Element'),
    ('novelty_knn_density_human_vs_allai.csv', 'Novelty-KNN'),
    ('bertopic_region_coverage_human_vs_allai.csv', 'BERTopic'),
    ('mesh_coverage_human_vs_allai.csv', 'MeSH'),
    ('style_features_human_vs_allai.csv', 'Style'),
]:
    _p = TABLES_DIR_AI / _csv_name
    if _p.exists():
        _tmp = pd.read_csv(_p)
        metric_col = 'metric' if 'metric' in _tmp.columns else \
                     ('feature' if 'feature' in _tmp.columns else _tmp.columns[0])
        for _, _r in _tmp.iterrows():
            summary_rows.append({
                'domain': _domain,
                'metric': _r.get(metric_col, ''),
                'human_val': _r.get('human_mean', np.nan),
                'ai_boot_mean': _r.get('ai_mean', np.nan),
                'perm_p': _r.get('mw_p', np.nan),
                'cliffs_delta': _r.get('cliffs_delta', np.nan),
                'sig': fmt_p(_r.get('mw_p', np.nan)) if not np.isnan(_r.get('mw_p', np.nan)) else 'n/a'
            })

summary_final = pd.DataFrame(summary_rows)
summary_final.to_csv(TABLES_DIR_AI / 'proposal_metrics_summary_human_vs_allai.csv', index=False)
print(f'Summary table: {len(summary_final)} rows across {summary_final["domain"].nunique()} domains')
print(summary_final.groupby('domain')['sig'].apply(lambda x: (x.isin(['***','**','*'])).sum()).reset_index(
    ).rename(columns={'sig': 'n_significant'}).to_string())
print('\nAll outputs saved to:', TABLES_DIR_AI)
print('All figures saved to:', FIGURES_DIR_AI)
print('\nNotebook complete.')
